# SHAP distributions for top-N features

In [1]:
# =======================
# Imports & configuration
# =======================
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, Layout, interactive_output
from IPython.display import display, clear_output

# --- datasets & paths ---
dataset_identifiers = ["Covid_code", "Diabetes_code", "Forest_code", "Mammography_code", "Satimage_code"]
base_path = "./"  #newResults directory
file_pattern = "sorted_orderValuesScore_*.csv"

DISPLAY_NAMES = {
    "Covid_code": "SMS",
    "Mammography_code": "Mammography",
    "Forest_code": "Forest",
    "Satimage_code": "Satimage",
    "Diabetes_code": "Diabetes"
}

IR_ORDER  = [1, 2, 3, 4, 5]
IR_LABELS = {1: "1:49", 2: "1:19", 3: "1:9", 4: "3:17", 5: "1:4"}
IR_COLORS = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c", 4: "#d62728", 5: "#9467bd"}

FIG_DPI = 600
FIG_FMT = "png"
OUT_DIR = os.path.join(base_path, "figs_TOP3_byFeature_violin_byMethod_2x2")
os.makedirs(OUT_DIR, exist_ok=True)

# =======================
# Helpers (same as original)
# =======================
def parse_ir_subset_covid(s: str):
    if not isinstance(s, str):
        return (np.nan, np.nan)
    m = re.search(r'ir[_\-]?(\d+)(?:.*?(?:subset|sub)[_\-]?(\d+))?', s, flags=re.IGNORECASE)
    if m:
        ir = int(m.group(1))
        sub = int(m.group(2)) if m.group(2) is not None else np.nan
        return (f"lr_shap_list_ir_{ir}", sub)
    return (np.nan, np.nan)

def extract_method_from_filename(path: str):
    fname = os.path.basename(path)
    m = re.match(r"sorted_orderValuesScore_(.+)\.csv$", fname, flags=re.IGNORECASE)
    return m.group(1) if m else "unknown"

def ir_str_to_int(ir_str: str):
    if not isinstance(ir_str, str):
        return np.nan
    m = re.search(r'ir[_\-]?(\d+)', ir_str, flags=re.IGNORECASE)
    return int(m.group(1)) if m else np.nan

def build_long(base_path: str, dataset_ids):
    recs = []
    for dataset in dataset_ids:
        ds_dir = os.path.join(base_path, dataset)
        csvs = glob.glob(os.path.join(ds_dir, file_pattern))
        if not csvs:
            print(f"[WARN] No files for {dataset}")
            continue

        for csv_path in csvs:
            method = extract_method_from_filename(csv_path)
            df = pd.read_csv(csv_path)
            if "Unnamed: 0" in df.columns:
                df = df.drop(columns=["Unnamed: 0"])
            df.columns = ['feature', 'shapMean', 'shapStd', 'shapSum', 'subset', 'imbalance_ratio']

            if dataset == "Covid_code":
                parsed = df['subset'].apply(parse_ir_subset_covid)
                df['imbalance_ratio'] = parsed.apply(lambda x: x[0])
                df['subset'] = parsed.apply(lambda x: x[1])

            df['IR'] = df['imbalance_ratio'].apply(ir_str_to_int)
            df = df[df['IR'].isin(IR_ORDER)]
            df['IR_label'] = df['IR'].map(IR_LABELS)

            keep = ['feature', 'subset', 'IR', 'IR_label', 'shapMean']
            dd = df[keep].copy()
            dd['dataset'] = dataset
            dd['method']  = method
            recs.append(dd)

    if not recs:
        raise RuntimeError("No data loaded.")
    return pd.concat(recs, ignore_index=True)

def topN_by_dataset(long_df: pd.DataFrame, N=3):
    top_map = {}
    for ds, g in long_df.groupby('dataset'):
        imp = (g.assign(a=g['shapMean'].abs())
                 .groupby('feature', as_index=False)['a'].mean()
                 .sort_values('a', ascending=False))
        top_map[ds] = imp['feature'].head(N).tolist()
    return top_map

# =======================
# Original plotting (Matplotlib 2x2), with small params
# =======================
def plot_dataset_grid_2x2_by_method_violin(long_df: pd.DataFrame, dataset: str,
                                           features, display_labels,
                                           width=0.12, gap_feat=0.35,
                                           show_y_range_in_title=False,
                                           save_to_file=False,
                                           unify_y_axis=False):  # <<< NEW
    g = long_df[long_df['dataset'] == dataset].copy()
    if g.empty:
        print(f"[WARN] No data for {dataset}")
        return

    methods = sorted(g['method'].unique())
    max_panels = 4
    methods = methods[:max_panels]
    n_panels = len(methods)
    n_rows, n_cols = 2, 2

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 8), sharey=False)
    axes = axes.flatten()

    n_irs = len(IR_ORDER)
    legend_handles = {}
    have_any_handle = False

    for ax_idx in range(n_rows * n_cols):
        ax = axes[ax_idx]
        if ax_idx >= n_panels:
            ax.set_axis_off()
            continue

        method = methods[ax_idx]
        gm = g[g['method'] == method]

        base_feat = np.arange(len(features)) * (n_irs * width + gap_feat)

        for iR, ir in enumerate(IR_ORDER):
            pos = base_feat + iR * width
            data = []
            for feat in features:
                vals = gm[(gm['feature'] == feat) & (gm['IR'] == ir)]['shapMean'].values
                data.append(vals if vals.size else np.array([np.nan]))

            vp = ax.violinplot(
                data,
                positions=pos,
                widths=width * 0.95,
                showmeans=False,
                showextrema=False,
                showmedians=False
            )
            for body in vp['bodies']:
                body.set_facecolor(IR_COLORS[ir])
                body.set_alpha(0.55)
                body.set_edgecolor("none")

            has_valid = any(np.isfinite(v).any() for v in data)
            if has_valid and ir not in legend_handles:
                legend_handles[ir] = vp['bodies'][0]
                have_any_handle = True

            meds = [np.nanmedian(v) if np.isfinite(np.nanmedian(v)) else np.nan for v in data]
            ax.scatter(pos, meds, s=12, zorder=3, edgecolors='k', linewidths=0.3)

        centers = base_feat + (n_irs - 1) * width / 2
        ax.set_xticks(centers)
        ax.set_xticklabels(display_labels, rotation=20, ha='right')

        if show_y_range_in_title:
            ymin, ymax = ax.get_ylim()
            ax.set_title(f"{method}\n(range {ymin:.2f}–{ymax:.2f})")
        else:
            ax.set_title(method)

        ax.grid(axis='y', alpha=0.3)
        if ax_idx % n_cols == 0:
            ax.set_ylabel("SHAP")

    # <<< NEW — ujednolicenie zakresów osi Y
    if unify_y_axis:
        active_axes = [axes[i] for i in range(n_panels)]
        if active_axes:
            gmin = min(ax.get_ylim()[0] for ax in active_axes)
            gmax = max(ax.get_ylim()[1] for ax in active_axes)
            for ax in active_axes:
                ax.set_ylim(gmin, gmax)

    dataset_display = DISPLAY_NAMES.get(dataset, dataset)
    fig.suptitle(f"{dataset_display} raw SHAP distributions by IR for Top-N features",
                 y=0.99, fontsize=13)

    if have_any_handle:
        handles = [legend_handles[i] for i in IR_ORDER if i in legend_handles]
        labels  = [IR_LABELS[i] for i in IR_ORDER if i in legend_handles]
        if handles and labels:
            fig.legend(handles, labels, title="IR (pos:neg)",
                       ncol=len(labels), loc="lower center",
                       bbox_to_anchor=(0.5, -0.05), frameon=True)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    if save_to_file:
        out_path = os.path.join(OUT_DIR, f"{dataset_display}__TOP3_byFeature__violin_byMethod_2x2.{FIG_FMT}")
        plt.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
        print(f"[INFO] Saved figure: {out_path}")

    plt.show()

# =======================
# Build data once
# =======================
long_df = build_long(base_path, dataset_identifiers)

# =======================
# Widgets UI
# =======================
w_ds = Dropdown(
    options=[(DISPLAY_NAMES.get(ds, ds), ds) for ds in dataset_identifiers],
    value=dataset_identifiers[0],
    description="Dataset:",
    layout=Layout(width="280px")
)
w_topn = IntSlider(value=3, min=1, max=6, step=1, description="Top-N:", continuous_update=False, layout=Layout(width="350px"))
w_save = Checkbox(value=False, description="Save to file")
w_showyr = Checkbox(value=False, description="Show Y-range in titles")
w_unifyy = Checkbox(value=False, description="Unified axis")  # <<< NEW
out = Output()

def recompute_and_plot(dataset, topn, save_to_file, show_y_range_in_title, unify_y_axis):  # <<< NEW
    out.clear_output(wait=True)
    with out:
        # compute top-N per dataset
        top_map = topN_by_dataset(long_df, N=topn)
        feats = top_map.get(dataset, [])
        if not feats:
            print(f"[WARN] No Top-{topn} for {dataset}")
            return

        # display labels (keep your Covid replacements)
        labels = feats.copy()
        if dataset == "Covid_code" and len(labels) >= 3:
            labels[1] = "SEX"
            labels[2] = "ARTERIAL_HIPERTENSION"

        plot_dataset_grid_2x2_by_method_violin(
            long_df, dataset,
            features=feats, display_labels=labels,
            width=0.12, gap_feat=0.35,
            show_y_range_in_title=show_y_range_in_title,
            save_to_file=save_to_file,
            unify_y_axis=unify_y_axis  # <<< NEW
        )

controls = {
    'dataset': w_ds,
    'topn': w_topn,
    'save_to_file': w_save,
    'show_y_range_in_title': w_showyr,
    'unify_y_axis': w_unifyy  
}
io = interactive_output(recompute_and_plot, controls)

ui = VBox([HBox([w_ds, w_topn, w_save, w_showyr, w_unifyy]), out])  # <<< NEW
display(ui)

# initial draw
recompute_and_plot(w_ds.value, w_topn.value, w_save.value, w_showyr.value, w_unifyy.value)  # <<< NEW


[WARN] No files for Covid_code


In [2]:
# =======================
# Imports & configuration
# =======================
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, Layout, interactive_output
from IPython.display import display, clear_output

# --- datasets & paths ---
dataset_identifiers = ["Covid_code", "Diabetes_code", "Forest_code", "Mammography_code", "Satimage_code"]
base_path = "./"  #"../"   # <- dostosuj w razie potrzeby
file_pattern = "sorted_orderValuesScore_*.csv"

DISPLAY_NAMES = {
    "Covid_code": "SMS",
    "Mammography_code": "Mammography",
    "Forest_code": "Forest",
    "Satimage_code": "Satimage",
    "Diabetes_code": "Diabetes"
}

IR_ORDER  = [1, 2, 3, 4, 5]
IR_LABELS = {1: "1:49", 2: "1:19", 3: "1:9", 4: "3:17", 5: "1:4"}
IR_COLORS = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c", 4: "#d62728", 5: "#9467bd"}

FIG_DPI = 600
FIG_FMT = "png"
OUT_DIR = os.path.join(base_path, "figs_TOP3_byFeature_violin_byMethod_2x2")
os.makedirs(OUT_DIR, exist_ok=True)

# =======================
# Helpers
# =======================
def parse_ir_subset_covid(s: str):
    if not isinstance(s, str):
        return (np.nan, np.nan)
    m = re.search(r'ir[_\-]?(\d+)(?:.*?(?:subset|sub)[_\-]?(\d+))?', s, flags=re.IGNORECASE)
    if m:
        ir = int(m.group(1))
        sub = int(m.group(2)) if m.group(2) is not None else np.nan
        return (f"lr_shap_list_ir_{ir}", sub)
    return (np.nan, np.nan)

def extract_method_from_filename(path: str):
    fname = os.path.basename(path)
    m = re.match(r"sorted_orderValuesScore_(.+)\.csv$", fname, flags=re.IGNORECASE)
    return m.group(1) if m else "unknown"

def ir_str_to_int(ir_str: str):
    if not isinstance(ir_str, str):
        return np.nan
    m = re.search(r'ir[_\-]?(\d+)', ir_str, flags=re.IGNORECASE)
    return int(m.group(1)) if m else np.nan

def build_long(base_path: str, dataset_ids):
    recs = []
    for dataset in dataset_ids:
        ds_dir = os.path.join(base_path, dataset)
        csvs = glob.glob(os.path.join(ds_dir, file_pattern))
        if not csvs:
            print(f"[WARN] No files for {dataset}")
            continue

        for csv_path in csvs:
            method = extract_method_from_filename(csv_path)
            df = pd.read_csv(csv_path)
            if "Unnamed: 0" in df.columns:
                df = df.drop(columns=["Unnamed: 0"])
            df.columns = ['feature', 'shapMean', 'shapStd', 'shapSum', 'subset', 'imbalance_ratio']

            if dataset == "Covid_code":
                parsed = df['subset'].apply(parse_ir_subset_covid)
                df['imbalance_ratio'] = parsed.apply(lambda x: x[0])
                df['subset'] = parsed.apply(lambda x: x[1])

            df['IR'] = df['imbalance_ratio'].apply(ir_str_to_int)
            df = df[df['IR'].isin(IR_ORDER)]
            df['IR_label'] = df['IR'].map(IR_LABELS)

            keep = ['feature', 'subset', 'IR', 'IR_label', 'shapMean']
            dd = df[keep].copy()
            dd['dataset'] = dataset
            dd['method']  = method
            recs.append(dd)

    if not recs:
        raise RuntimeError("No data loaded.")
    return pd.concat(recs, ignore_index=True)

def topN_by_dataset(long_df: pd.DataFrame, N=3):
    top_map = {}
    for ds, g in long_df.groupby('dataset'):
        imp = (g.assign(a=g['shapMean'].abs())
                 .groupby('feature', as_index=False)['a'].mean()
                 .sort_values('a', ascending=False))
        top_map[ds] = imp['feature'].head(N).tolist()
    return top_map

# =======================
# Plot: 1x4 grid (vertical), z opcją Unified axis
# =======================
def plot_dataset_grid_2x2_by_method_violin(long_df: pd.DataFrame, dataset: str,
                                           features, display_labels,
                                           width=0.12, gap_feat=0.35,
                                           show_y_range_in_title=False,
                                           save_to_file=False,
                                           unify_y_axis=False):
    g = long_df[long_df['dataset'] == dataset].copy()
    if g.empty:
        print(f"[WARN] No data for {dataset}")
        return

    methods = sorted(g['method'].unique())[:4]
    n_panels = len(methods)

    # Układ 1x4
    fig, axes = plt.subplots(4, 1, figsize=(12, 16), sharey=False)
    axes = np.ravel(axes)  # zawsze tablica

    n_irs = len(IR_ORDER)
    legend_handles = {}

    for ax_idx, ax in enumerate(axes):
        if ax_idx >= n_panels:
            ax.set_axis_off()
            continue

        method = methods[ax_idx]
        gm = g[g['method'] == method]
        base_feat = np.arange(len(features)) * (n_irs * width + gap_feat)

        for ir in IR_ORDER:
            pos = base_feat + IR_ORDER.index(ir) * width
            data = []
            for feat in features:
                vals = gm[(gm['feature'] == feat) & (gm['IR'] == ir)]['shapMean'].values
                data.append(vals if vals.size else np.array([np.nan]))

            vp = ax.violinplot(
                data,
                positions=pos,
                widths=width * 0.95,
                showmeans=False,
                showextrema=False,
                showmedians=False
            )
            for body in vp['bodies']:
                body.set_facecolor(IR_COLORS[ir])
                body.set_alpha(0.55)
                body.set_edgecolor("none")

            # uchwyt do legendy tylko raz na IR
            if ir not in legend_handles and any(np.isfinite(v).any() for v in data):
                legend_handles[ir] = vp['bodies'][0]

            meds = [np.nanmedian(v) if np.isfinite(np.nanmedian(v)) else np.nan for v in data]
            ax.scatter(pos, meds, s=12, zorder=3, edgecolors='k', linewidths=0.3)

        centers = base_feat + (n_irs - 1) * width / 2
        ax.set_xticks(centers)
        ax.set_xticklabels(display_labels, rotation=20, ha='right')

        if show_y_range_in_title:
            ymin, ymax = ax.get_ylim()
            ax.set_title(f"{method}\n(range {ymin:.2f}–{ymax:.2f})")
        else:
            ax.set_title(method)

        ax.grid(axis='y', alpha=0.3)
        ax.set_ylabel("SHAP")

    # Unified axis: wspólne min/max po narysowaniu paneli
    if unify_y_axis and n_panels > 0:
        active_axes = axes[:n_panels]
        gmin = min(ax.get_ylim()[0] for ax in active_axes)
        gmax = max(ax.get_ylim()[1] for ax in active_axes)
        for ax in active_axes:
            ax.set_ylim(gmin, gmax)

    dataset_display = DISPLAY_NAMES.get(dataset, dataset)
    fig.suptitle(f"{dataset_display} raw SHAP distributions by IR for Top-N features",
                 y=0.99, fontsize=13)

    if legend_handles:
        handles = [legend_handles[i] for i in IR_ORDER if i in legend_handles]
        labels  = [IR_LABELS[i] for i in IR_ORDER if i in legend_handles]
        if handles:
            fig.legend(handles, labels, title="IR (pos:neg)",
                       ncol=len(handles), loc="lower center",
                       bbox_to_anchor=(0.5, -0.02), frameon=True)

    plt.tight_layout(rect=[0, 0.03, 1, 0.96])

    if save_to_file:
        out_path = os.path.join(OUT_DIR, f"{dataset_display}__TOP3_byFeature__violin_byMethod_2x2.{FIG_FMT}")
        plt.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
        print(f"[INFO] Saved figure: {out_path}")

    plt.show()

# =======================
# Build data once
# =======================
long_df = build_long(base_path, dataset_identifiers)

# =======================
# Widgets UI
# =======================
w_ds = Dropdown(
    options=[(DISPLAY_NAMES.get(ds, ds), ds) for ds in dataset_identifiers],
    value=dataset_identifiers[0],
    description="Dataset:",
    layout=Layout(width="280px")
)
w_topn = IntSlider(value=3, min=1, max=6, step=1, description="Top-N:", continuous_update=False, layout=Layout(width="350px"))
w_save = Checkbox(value=False, description="Save to file")
w_showyr = Checkbox(value=False, description="Show Y-range in titles")
w_unifyy = Checkbox(value=False, description="Unified axis")
out = Output()

def recompute_and_plot(dataset, topn, save_to_file, show_y_range_in_title, unify_y_axis):
    out.clear_output(wait=True)
    with out:
        top_map = topN_by_dataset(long_df, N=topn)
        feats = top_map.get(dataset, [])
        if not feats:
            print(f"[WARN] No Top-{topn} for {dataset}")
            return

        labels = feats.copy()
        if dataset == "Covid_code" and len(labels) >= 3:
            labels[1] = "SEX"
            labels[2] = "ARTERIAL_HIPERTENSION"

        plot_dataset_grid_2x2_by_method_violin(
            long_df, dataset,
            features=feats, display_labels=labels,
            width=0.12, gap_feat=0.35,
            show_y_range_in_title=show_y_range_in_title,
            save_to_file=save_to_file,
            unify_y_axis=unify_y_axis
        )

controls = {
    'dataset': w_ds,
    'topn': w_topn,
    'save_to_file': w_save,
    'show_y_range_in_title': w_showyr,
    'unify_y_axis': w_unifyy
}
interactive_output(recompute_and_plot, controls)  # podpięcie callbacka

ui = VBox([HBox([w_ds, w_topn, w_save, w_showyr, w_unifyy]), out])
display(ui)

# initial draw
recompute_and_plot(w_ds.value, w_topn.value, w_save.value, w_showyr.value, w_unifyy.value)


[WARN] No files for Covid_code


# Histograms of unique top-N feature sets

In [3]:
%%time
# ============================================
# Histograms of unique Top-N feature sets per IR
# + model selector, IR-colored bars, proper sliders for Top-N & Bins
# + 2-wierszowy dashboard (Dataset/Model w 1. linii, Top-N/Bins w 2. linii)
# ============================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, Layout, interactive_output
from IPython.display import display

# Output dir
OUT_DIR_HIST = os.path.join(base_path, "figs_TOPsets_hist_byIR")
os.makedirs(OUT_DIR_HIST, exist_ok=True)

# --------- helpers ---------
def compute_unique_top_sets(df: pd.DataFrame, dataset: str, topn: int = 5, model: str | None = None) -> pd.DataFrame:
    g = df[df['dataset'] == dataset].copy()
    if model is not None:
        g = g[g['method'] == model]
    if g.empty:
        return pd.DataFrame(columns=["IR","IR_label","top_set","count"])

    g['subset_filled'] = g['subset'].fillna(-1)
    topn = int(max(1, min(10, topn)))  # clamp to 1..10 (top-N = upper limit)

    recs = []
    for (_, subset_val, ir), sub in g.groupby(['method','subset_filled','IR'], dropna=False):
        imp = (sub.assign(a=sub['shapMean'].abs())
                 .groupby('feature', as_index=False)['a'].mean()
                 .sort_values('a', ascending=False))
        k = min(topn, len(imp))            # if fewer features exist, take what's available
        if k == 0: 
            continue
        top_feats = tuple(imp['feature'].head(k))
        recs.append({"IR": ir, "top_set": top_feats})

    if not recs:
        return pd.DataFrame(columns=["IR","IR_label","top_set","count"])

    counts = (pd.DataFrame(recs)
                .groupby(['IR','top_set']).size().reset_index(name='count'))
    counts['IR_label'] = counts['IR'].map(IR_LABELS)
    counts['IR_order_idx'] = counts['IR'].apply(lambda x: IR_ORDER.index(x) if x in IR_ORDER else 999)
    return counts.sort_values(['IR_order_idx','count'], ascending=[True,False]).drop(columns='IR_order_idx')

def _unified_bins_ylim(all_vals, bins: int):
    if not all_vals: return None, None
    concat = np.concatenate(all_vals)
    edges = np.histogram_bin_edges(concat, bins=int(bins))
    ymax = 0
    for v in all_vals:
        h, _ = np.histogram(v, bins=edges)
        ymax = max(ymax, int(h.max()) if h.size else 0)
    return edges, ymax

def plot_histograms(counts_df: pd.DataFrame, dataset: str, model: str | None,
                    bins: int = 20, save_to_file: bool = False, unified_axes: bool = False):
    if counts_df.empty:
        print(f"[WARN] No data to plot for: {dataset}" + (f" | model: {model}" if model else ""))
        return

    dataset_display = DISPLAY_NAMES.get(dataset, dataset)
    present_irs = [ir for ir in IR_ORDER if ir in counts_df['IR'].unique()]

    edges = y_max = None
    if unified_axes:
        all_vals = [counts_df[counts_df['IR']==ir]['count'].to_numpy() for ir in present_irs]
        edges, y_max = _unified_bins_ylim(all_vals, bins=bins)

    for ir in present_irs:
        vals = counts_df[counts_df['IR']==ir]['count'].to_numpy()
        ir_label = IR_LABELS.get(ir, str(ir))
        use_bins = edges if (unified_axes and edges is not None) else int(bins)
        color = IR_COLORS.get(ir, None)

        plt.figure(figsize=(6,4))
        plt.hist(vals, bins=use_bins, edgecolor='black', color=color)
        title_model = f" — {model}" if model else ""
        plt.title(f"{dataset_display}{title_model} — Histogram of unique Top-N sets (IR {ir_label})")
        plt.xlabel("Occurrences of Top-N feature set in (method × subset)")
        plt.ylabel("Number of unique Top-N sets")
        plt.grid(True, alpha=0.3)
        if unified_axes and edges is not None:
            plt.xlim(edges[0], edges[-1]); plt.ylim(0, max(1, y_max)*1.05)
        plt.tight_layout(); plt.show()

        if save_to_file:
            safe_ir = ir_label.replace(':','-')
            suffix  = "unified" if unified_axes else "auto"
            bins_n  = (len(edges)-1) if isinstance(use_bins, np.ndarray) else use_bins
            fn = f"{dataset_display}__{(model or 'all')}__TopSetsHist_IR{safe_ir}_bins{bins_n}_{suffix}.{FIG_FMT}"
            out_path = os.path.join(OUT_DIR_HIST, fn)

            plt.figure(figsize=(6,4))
            plt.hist(vals, bins=use_bins, edgecolor='black', color=color)
            plt.title(f"{dataset_display}{title_model} — Histogram of unique Top-N sets (IR {ir_label})")
            plt.xlabel("Occurrences of Top-N feature set in (method × subset)")
            plt.ylabel("Number of unique Top-N sets")
            plt.grid(True, alpha=0.3)
            if unified_axes and edges is not None:
                plt.xlim(edges[0], edges[-1]); plt.ylim(0, max(1, y_max)*1.05)
            plt.tight_layout(); plt.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight"); plt.close()
            print(f"[INFO] Saved: {out_path}")

# --------- widgets ---------
w_ds = Dropdown(
    options=[(DISPLAY_NAMES.get(ds, ds), ds) for ds in dataset_identifiers],
    value=dataset_identifiers[0], description="Dataset:",
    layout=Layout(width="260px")
)

w_model = Dropdown(options=[], value=None, description="Model:", layout=Layout(width="200px"))

# SUWAKI – wyraźny tor i odczyt po prawej
w_topn = IntSlider(
    value=5, min=1, max=5, step=1,
    description="Top-N:",
    readout=True, readout_format='d',
    continuous_update=False,
    layout=Layout(width="350px")
)
w_bins = IntSlider(
    value=20, min=5, max=80, step=1,
    description="Bins:",
    readout=True, readout_format='d',
    continuous_update=False,
    layout=Layout(width="350px")
)

w_unified = Checkbox(value=False, description="Unified axes")
w_save    = Checkbox(value=False, description="Save to file")
out       = Output()

# ładniejsze wyrównanie etykiet
w_topn.style = {"description_width": "initial"}
w_bins.style = {"description_width": "initial"}

def _refresh_models(ds_value):
    sub = long_df[long_df['dataset']==ds_value]
    methods = sorted(sub['method'].dropna().unique().tolist())
    options = [(m, m) for m in methods] if methods else [("—", None)]
    w_model.options = options
    w_model.value   = options[0][1] if options else None

_refresh_models(w_ds.value)
w_ds.observe(lambda ch: _refresh_models(ch['new']) if ch['name']=='value' and ch['new']!=ch['old'] else None)

def _recompute(dataset, topn, bins, save_to_file, unified_axes, model):
    out.clear_output(wait=True)
    with out:
        counts_df = compute_unique_top_sets(long_df, dataset, topn=topn, model=model)
        plot_histograms(counts_df, dataset, model, bins=bins,
                        save_to_file=save_to_file, unified_axes=unified_axes)

controls = {
    'dataset': w_ds, 'topn': w_topn, 'bins': w_bins,
    'save_to_file': w_save, 'unified_axes': w_unified, 'model': w_model
}

# --- układ w 2 liniach ---
row1 = HBox([w_ds, w_model], layout=Layout(align_items="center", gap="16px"))
row2 = HBox([w_topn, w_bins, w_unified, w_save], layout=Layout(align_items="center", gap="16px"))
io = interactive_output(_recompute, controls)

display(VBox([row1, row2, out], layout=Layout(gap="8px")))

# initial draw
_recompute(w_ds.value, w_topn.value, w_bins.value, w_save.value, w_unified.value, w_model.value)


CPU times: total: 62.5 ms
Wall time: 62.2 ms


In [4]:
%%time
# ============================================
# Violin plots of unique Top-N feature-set counts across all IRs
# Grid 2x2 for all models, optional unified Y-axis + legend at bottom
# ============================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, Layout, interactive_output
from IPython.display import display
from matplotlib.patches import Patch

# Output directory
OUT_DIR_VIOLIN = os.path.join(base_path, "figs_TOPsets_violin_allIR_grid")
os.makedirs(OUT_DIR_VIOLIN, exist_ok=True)

# --------- helpers ---------
def compute_unique_top_sets(df: pd.DataFrame, dataset: str, topn: int = 5, model: str | None = None) -> pd.DataFrame:
    """
    Count unique Top-N feature-sets in (method × subset).
    Returns DataFrame: [IR, IR_label, top_set, count].
    """
    g = df[df['dataset'] == dataset].copy()
    if model is not None:
        g = g[g['method'] == model]
    if g.empty:
        return pd.DataFrame(columns=["IR","IR_label","top_set","count"])

    g['subset_filled'] = g['subset'].fillna(-1)
    topn = int(max(1, min(10, topn)))  # clamp 1..10

    records = []
    for (_, subset_val, ir), sub in g.groupby(['method','subset_filled','IR'], dropna=False):
        imp = (sub.assign(a=sub['shapMean'].abs())
                 .groupby('feature', as_index=False)['a'].mean()
                 .sort_values('a', ascending=False))
        k = min(topn, len(imp))
        if k == 0:
            continue
        top_feats = tuple(imp['feature'].head(k))
        records.append({"IR": ir, "top_set": top_feats})

    if not records:
        return pd.DataFrame(columns=["IR","IR_label","top_set","count"])

    counts = (pd.DataFrame(records)
                .groupby(['IR','top_set']).size().reset_index(name='count'))
    counts['IR_label'] = counts['IR'].map(IR_LABELS)
    counts['IR_order_idx'] = counts['IR'].apply(lambda x: IR_ORDER.index(x) if x in IR_ORDER else 999)
    return counts.sort_values(['IR_order_idx','count'], ascending=[True,False]).drop(columns='IR_order_idx')

def _collect_violin_data(counts_df: pd.DataFrame):
    """Prepare data arrays, tick labels, and colors for a violin plot."""
    present_irs = [ir for ir in IR_ORDER if ir in counts_df['IR'].unique()]
    data_by_ir, tick_labels, body_colors = [], [], []
    for ir in present_irs:
        vals = counts_df[counts_df['IR'] == ir]['count'].to_numpy()
        if vals.size == 0:
            continue
        data_by_ir.append(vals)
        tick_labels.append(IR_LABELS.get(ir, str(ir)))
        body_colors.append(IR_COLORS.get(ir, None))
    return data_by_ir, tick_labels, body_colors, present_irs

def _draw_single_violin(ax, data_by_ir, tick_labels, body_colors, title_text):
    """Draw one violin plot into a given Axes."""
    parts = ax.violinplot(
        dataset=data_by_ir,
        showmeans=True,
        showmedians=True,
        showextrema=True
    )
    # color per IR (if available)
    for i, b in enumerate(parts['bodies']):
        col = body_colors[i]
        if col is not None:
            b.set_facecolor(col)
        b.set_alpha(0.7)
        b.set_edgecolor('black')
        b.set_linewidth(0.8)
    for k in ('cmeans', 'cmedians', 'cbars', 'cmins', 'cmaxes'):
        if k in parts:
            parts[k].set_linewidth(1.2)
            parts[k].set_color('black')

    ax.set_xticks(np.arange(1, len(tick_labels) + 1))
    ax.set_xticklabels(tick_labels)
    ax.set_title(title_text, fontsize=11)
    ax.set_xlabel("IR")
    ax.set_ylabel("Unique Top-N set count")
    ax.grid(True, axis='y', alpha=0.3)

def plot_violin_grid(counts_by_model: dict, dataset: str, unified_axis: bool = False, save_to_file: bool = False):
    """
    Draw a 2x2 grid of violin plots for up to 4 models.
    If unified_axis=True, set a common Y limit across all subplots.
    Add legend at bottom.
    """
    dataset_display = DISPLAY_NAMES.get(dataset, dataset)
    model_names = list(counts_by_model.keys())

    # Prepare per-model data
    prepared = {}
    global_max = 0
    all_present_irs = set()

    for m in model_names:
        cdf = counts_by_model[m]
        if cdf is None or cdf.empty:
            prepared[m] = None
            continue
        data_by_ir, tick_labels, body_colors, present_irs = _collect_violin_data(cdf)
        if not data_by_ir:
            prepared[m] = None
            continue
        prepared[m] = (data_by_ir, tick_labels, body_colors)
        all_present_irs.update(present_irs)

        # track global Y max for unified axis
        vals_concat = np.concatenate(data_by_ir)
        if vals_concat.size:
            global_max = max(global_max, vals_concat.max())

    # Create 2x2 grid
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()

    for i, ax in enumerate(axes):
        if i < len(model_names):
            m = model_names[i]
            content = prepared[m]
            if content is None:
                ax.set_title(f"{m} — no data")
                ax.axis('off')
                continue
            data_by_ir, tick_labels, body_colors = content
            _draw_single_violin(ax, data_by_ir, tick_labels, body_colors, title_text=m)
        else:
            ax.axis('off')

    # Unified Y axis
    if unified_axis and global_max > 0:
        ymax = max(1, int(global_max)) * 1.05
        for ax in axes:
            if ax.has_data():
                ax.set_ylim(0, ymax)

    fig.suptitle(f"{dataset_display} — Distributions of unique Top-N sets across IRs (all models)", fontsize=14)

    # -------- LEGEND AT BOTTOM --------
    legend_handles = []
    for ir in IR_ORDER:
        if ir in all_present_irs:
            legend_handles.append(
                Patch(facecolor=IR_COLORS.get(ir, 'gray'), label=IR_LABELS.get(ir, str(ir)))
            )

    fig.legend(
        handles=legend_handles,
        loc='lower center',
        ncol=min(len(legend_handles), 6),
        bbox_to_anchor=(0.5, -0.03)
    )

    fig.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.show()

    if save_to_file:
        fn = f"{dataset_display}__TopSets_Violin_AllIR_Grid.{FIG_FMT}"
        out_path = os.path.join(OUT_DIR_VIOLIN, fn)
        fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)
        print(f"[INFO] Saved: {out_path}")

# --------- widgets ---------
w_ds = Dropdown(
    options=[(DISPLAY_NAMES.get(ds, ds), ds) for ds in dataset_identifiers],
    value=dataset_identifiers[0],
    description="Dataset:",
    layout=Layout(width="260px")
)

w_topn = IntSlider(
    value=5, min=1, max=5, step=1,
    description="Top-N:",
    readout=True, readout_format='d',
    continuous_update=False,
    layout=Layout(width="350px")
)

w_unified = Checkbox(value=False, description="Unified axis")
w_save = Checkbox(value=False, description="Save to file")
out = Output()

w_topn.style = {"description_width": "initial"}

def _list_models_for_dataset(ds_value):
    sub = long_df[long_df['dataset'] == ds_value]
    return sorted(sub['method'].dropna().unique().tolist())

def _recompute(dataset, topn, unified_axis, save_to_file):
    out.clear_output(wait=True)
    with out:
        models = _list_models_for_dataset(dataset)
        if not models:
            print(f"[WARN] No models found for dataset: {dataset}")
            return
        models = models[:4]
        counts_by_model = {m: compute_unique_top_sets(long_df, dataset, topn=topn, model=m) for m in models}
        plot_violin_grid(counts_by_model, dataset, unified_axis=unified_axis, save_to_file=save_to_file)

controls = {
    'dataset': w_ds,
    'topn': w_topn,
    'unified_axis': w_unified,
    'save_to_file': w_save
}

row1 = HBox([w_ds], layout=Layout(align_items="center", gap="16px"))
row2 = HBox([w_topn, w_unified, w_save], layout=Layout(align_items="center", gap="16px"))
io = interactive_output(_recompute, controls)

display(VBox([row1, row2, out], layout=Layout(gap="8px")))

# initial draw
_recompute(w_ds.value, w_topn.value, w_unified.value, w_save.value)


CPU times: total: 31.2 ms
Wall time: 43 ms


In [5]:
%%time
# ============================================
# Violin plots of unique Top-N feature-set counts across all IRs
# Vertical layout 1x4 for all models, optional unified Y-axis + bottom legend
# ============================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, Layout, interactive_output
from IPython.display import display
from matplotlib.patches import Patch

# Output directory
OUT_DIR_VIOLIN = os.path.join(base_path, "figs_TOPsets_violin_allIR_grid")
os.makedirs(OUT_DIR_VIOLIN, exist_ok=True)

# --------- helpers ---------
def compute_unique_top_sets(df: pd.DataFrame, dataset: str, topn: int = 5, model: str | None = None) -> pd.DataFrame:
    g = df[df['dataset'] == dataset].copy()
    if model is not None:
        g = g[g['method'] == model]
    if g.empty:
        return pd.DataFrame(columns=["IR","IR_label","top_set","count"])

    g['subset_filled'] = g['subset'].fillna(-1)
    topn = int(max(1, min(10, topn)))

    records = []
    for (_, subset_val, ir), sub in g.groupby(['method','subset_filled','IR'], dropna=False):
        imp = (sub.assign(a=sub['shapMean'].abs())
                 .groupby('feature', as_index=False)['a'].mean()
                 .sort_values('a', ascending=False))
        k = min(topn, len(imp))
        if k == 0:
            continue
        top_feats = tuple(imp['feature'].head(k))
        records.append({"IR": ir, "top_set": top_feats})

    if not records:
        return pd.DataFrame(columns=["IR","IR_label","top_set","count"])

    counts = (pd.DataFrame(records)
                .groupby(['IR','top_set']).size().reset_index(name='count'))
    counts['IR_label'] = counts['IR'].map(IR_LABELS)
    counts['IR_order_idx'] = counts['IR'].apply(lambda x: IR_ORDER.index(x) if x in IR_ORDER else 999)
    return counts.sort_values(['IR_order_idx','count'], ascending=[True,False]).drop(columns='IR_order_idx')

def _collect_violin_data(counts_df: pd.DataFrame):
    present_irs = [ir for ir in IR_ORDER if ir in counts_df['IR'].unique()]
    data_by_ir, tick_labels, body_colors = [], [], []
    for ir in present_irs:
        vals = counts_df[counts_df['IR'] == ir]['count'].to_numpy()
        if vals.size == 0:
            continue
        data_by_ir.append(vals)
        tick_labels.append(IR_LABELS.get(ir, str(ir)))
        body_colors.append(IR_COLORS.get(ir, None))
    return data_by_ir, tick_labels, body_colors, present_irs

def _draw_single_violin(ax, data_by_ir, tick_labels, body_colors, title_text):
    parts = ax.violinplot(
        dataset=data_by_ir,
        showmeans=True,
        showmedians=True,
        showextrema=True
    )
    for i, b in enumerate(parts['bodies']):
        col = body_colors[i]
        if col is not None:
            b.set_facecolor(col)
        b.set_alpha(0.7)
        b.set_edgecolor('black')
        b.set_linewidth(0.8)
    for k in ('cmeans', 'cmedians', 'cbars', 'cmins', 'cmaxes'):
        if k in parts:
            parts[k].set_linewidth(1.2)
            parts[k].set_color('black')

    ax.set_xticks(np.arange(1, len(tick_labels) + 1))
    ax.set_xticklabels(tick_labels)
    ax.set_title(title_text, fontsize=11)
    ax.set_xlabel("IR")
    ax.set_ylabel("Unique Top-N set count")
    ax.grid(True, axis='y', alpha=0.3)

def plot_violin_grid(counts_by_model: dict, dataset: str, unified_axis: bool = False, save_to_file: bool = False):
    dataset_display = DISPLAY_NAMES.get(dataset, dataset)
    model_names = list(counts_by_model.keys())

    prepared = {}
    global_max = 0
    all_present_irs = set()

    for m in model_names:
        cdf = counts_by_model[m]
        if cdf is None or cdf.empty:
            prepared[m] = None
            continue
        data_by_ir, tick_labels, body_colors, present_irs = _collect_violin_data(cdf)
        if not data_by_ir:
            prepared[m] = None
            continue
        prepared[m] = (data_by_ir, tick_labels, body_colors)
        all_present_irs.update(present_irs)

        vals_concat = np.concatenate(data_by_ir)
        if vals_concat.size:
            global_max = max(global_max, vals_concat.max())

    # ======== 1×4 layout ========
    fig, axes = plt.subplots(4, 1, figsize=(10, 16))
    axes = axes.flatten()

    for i, ax in enumerate(axes):
        if i < len(model_names):
            m = model_names[i]
            content = prepared[m]
            if content is None:
                ax.set_title(f"{m} — no data")
                ax.axis('off')
                continue
            data_by_ir, tick_labels, body_colors = content
            _draw_single_violin(ax, data_by_ir, tick_labels, body_colors, title_text=m)
        else:
            ax.axis('off')

    if unified_axis and global_max > 0:
        ymax = max(1, int(global_max)) * 1.05
        for ax in axes:
            if ax.has_data():
                ax.set_ylim(0, ymax)

    fig.suptitle(f"{dataset_display} — Distributions of unique Top-N sets across IRs (all models)", fontsize=14)

    legend_handles = []
    for ir in IR_ORDER:
        if ir in all_present_irs:
            legend_handles.append(
                Patch(facecolor=IR_COLORS.get(ir, 'gray'), label=IR_LABELS.get(ir, str(ir)))
            )

    fig.legend(
        handles=legend_handles,
        loc='lower center',
        ncol=min(len(legend_handles), 6),
        bbox_to_anchor=(0.5, -0.02)
    )

    fig.tight_layout(rect=[0, 0.04, 1, 0.95])
    plt.show()

    if save_to_file:
        fn = f"{dataset_display}__TopSets_Violin_AllIR_VerticalGrid.{FIG_FMT}"
        out_path = os.path.join(OUT_DIR_VIOLIN, fn)
        fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)
        print(f"[INFO] Saved: {out_path}")

# --------- widgets ---------
w_ds = Dropdown(
    options=[(DISPLAY_NAMES.get(ds, ds), ds) for ds in dataset_identifiers],
    value=dataset_identifiers[0],
    description="Dataset:",
    layout=Layout(width="260px")
)

w_topn = IntSlider(
    value=5, min=1, max=5, step=1,
    description="Top-N:",
    readout=True, readout_format='d',
    continuous_update=False,
    layout=Layout(width="350px")
)

w_unified = Checkbox(value=False, description="Unified axis")
w_save = Checkbox(value=False, description="Save to file")
out = Output()

w_topn.style = {"description_width": "initial"}

def _list_models_for_dataset(ds_value):
    sub = long_df[long_df['dataset'] == ds_value]
    return sorted(sub['method'].dropna().unique().tolist())

def _recompute(dataset, topn, unified_axis, save_to_file):
    out.clear_output(wait=True)
    with out:
        models = _list_models_for_dataset(dataset)
        if not models:
            print(f"[WARN] No models found for dataset: {dataset}")
            return
        models = models[:4]
        counts_by_model = {m: compute_unique_top_sets(long_df, dataset, topn=topn, model=m) for m in models}
        plot_violin_grid(counts_by_model, dataset, unified_axis=unified_axis, save_to_file=save_to_file)

controls = {
    'dataset': w_ds,
    'topn': w_topn,
    'unified_axis': w_unified,
    'save_to_file': w_save
}

row1 = HBox([w_ds], layout=Layout(align_items="center", gap="16px"))
row2 = HBox([w_topn, w_unified, w_save], layout=Layout(align_items="center", gap="16px"))
io = interactive_output(_recompute, controls)

display(VBox([row1, row2, out], layout=Layout(gap="8px")))

# initial draw
_recompute(w_ds.value, w_topn.value, w_unified.value, w_save.value)


CPU times: total: 31.2 ms
Wall time: 36.4 ms


# Distribution of unique top-N features

In [6]:
%%time
# ============================================
# Violin plots (1x4 vertical) of unique Top-N feature-set counts across all IRs
# + optional unified Y-axis, bottom legend
# + optional line chart: repetition of unique Top-N feature sets per IR (from unique_feature_lists_df)
# ============================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, Layout, interactive_output
from IPython.display import display
from matplotlib.patches import Patch
import math

# ----- CONFIG -----
OUT_DIR_VIOLIN = os.path.join(base_path, "figs_TOPsets_violin_allIR_grid")
os.makedirs(OUT_DIR_VIOLIN, exist_ok=True)
USE_IR_COLORS = True  # set to False to use rank_colors fallback like in your snippet

# Fallback palette identical to your snippet (only used when USE_IR_COLORS=False)
rank_colors = {1: 'blue', 2: 'green', 3: 'yellow', 4: 'orange', 5: 'red'}

# --------- helpers for violin ---------
def compute_unique_top_sets(df: pd.DataFrame, dataset: str, topn: int = 5, model: str | None = None) -> pd.DataFrame:
    g = df[df['dataset'] == dataset].copy()
    if model is not None:
        g = g[g['method'] == model]
    if g.empty:
        return pd.DataFrame(columns=["IR","IR_label","top_set","count"])

    g['subset_filled'] = g['subset'].fillna(-1)
    topn = int(max(1, min(10, topn)))

    records = []
    for (_, subset_val, ir), sub in g.groupby(['method','subset_filled','IR'], dropna=False):
        imp = (sub.assign(a=sub['shapMean'].abs())
                 .groupby('feature', as_index=False)['a'].mean()
                 .sort_values('a', ascending=False))
        k = min(topn, len(imp))
        if k == 0:
            continue
        top_feats = tuple(imp['feature'].head(k))
        records.append({"IR": ir, "top_set": top_feats})

    if not records:
        return pd.DataFrame(columns=["IR","IR_label","top_set","count"])

    counts = (pd.DataFrame(records)
                .groupby(['IR','top_set']).size().reset_index(name='count'))
    counts['IR_label'] = counts['IR'].map(IR_LABELS)
    counts['IR_order_idx'] = counts['IR'].apply(lambda x: IR_ORDER.index(x) if x in IR_ORDER else 999)
    return counts.sort_values(['IR_order_idx','count'], ascending=[True,False]).drop(columns='IR_order_idx')

def _collect_violin_data(counts_df: pd.DataFrame):
    present_irs = [ir for ir in IR_ORDER if ir in counts_df['IR'].unique()]
    data_by_ir, tick_labels, body_colors = [], [], []
    for ir in present_irs:
        vals = counts_df[counts_df['IR'] == ir]['count'].to_numpy()
        if vals.size == 0:
            continue
        data_by_ir.append(vals)
        tick_labels.append(IR_LABELS.get(ir, str(ir)))
        body_colors.append(IR_COLORS.get(ir, None))
    return data_by_ir, tick_labels, body_colors, present_irs

def _draw_single_violin(ax, data_by_ir, tick_labels, body_colors, title_text):
    parts = ax.violinplot(
        dataset=data_by_ir,
        showmeans=True,
        showmedians=True,
        showextrema=True
    )
    for i, b in enumerate(parts['bodies']):
        col = body_colors[i]
        if col is not None:
            b.set_facecolor(col)
        b.set_alpha(0.7)
        b.set_edgecolor('black')
        b.set_linewidth(0.8)
    for k in ('cmeans', 'cmedians', 'cbars', 'cmins', 'cmaxes'):
        if k in parts:
            parts[k].set_linewidth(1.2)
            parts[k].set_color('black')

    ax.set_xticks(np.arange(1, len(tick_labels) + 1))
    ax.set_xticklabels(tick_labels)
    ax.set_title(title_text, fontsize=11)
    ax.set_xlabel("IR")
    ax.set_ylabel("Unique Top-N set count")
    ax.grid(True, axis='y', alpha=0.3)

def plot_violin_grid(counts_by_model: dict, dataset: str, unified_axis: bool = False, save_to_file: bool = False):
    dataset_display = DISPLAY_NAMES.get(dataset, dataset)
    model_names = list(counts_by_model.keys())

    prepared = {}
    global_max = 0
    all_present_irs = set()

    for m in model_names:
        cdf = counts_by_model[m]
        if cdf is None or cdf.empty:
            prepared[m] = None
            continue
        data_by_ir, tick_labels, body_colors, present_irs = _collect_violin_data(cdf)
        if not data_by_ir:
            prepared[m] = None
            continue
        prepared[m] = (data_by_ir, tick_labels, body_colors)
        all_present_irs.update(present_irs)

        vals_concat = np.concatenate(data_by_ir)
        if vals_concat.size:
            global_max = max(global_max, vals_concat.max())

    # 1×4 vertical layout
    fig, axes = plt.subplots(4, 1, figsize=(10, 16))
    axes = axes.flatten()

    for i, ax in enumerate(axes):
        if i < len(model_names):
            m = model_names[i]
            content = prepared[m]
            if content is None:
                ax.set_title(f"{m} — no data")
                ax.axis('off')
                continue
            data_by_ir, tick_labels, body_colors = content
            _draw_single_violin(ax, data_by_ir, tick_labels, body_colors, title_text=m)
        else:
            ax.axis('off')

    if unified_axis and global_max > 0:
        ymax = max(1, int(global_max)) * 1.05
        for ax in axes:
            if ax.has_data():
                ax.set_ylim(0, ymax)

    fig.suptitle(f"{dataset_display} — Distributions of unique Top-N sets across IRs (all models)", fontsize=14)

    legend_handles = []
    for ir in IR_ORDER:
        if ir in all_present_irs:
            legend_handles.append(
                Patch(facecolor=IR_COLORS.get(ir, 'gray'), label=IR_LABELS.get(ir, str(ir)))
            )

    fig.legend(
        handles=legend_handles,
        loc='lower center',
        ncol=min(len(legend_handles), 6),
        bbox_to_anchor=(0.5, -0.02)
    )

    fig.tight_layout(rect=[0, 0.04, 1, 0.95])
    plt.show()

    if save_to_file:
        fn = f"{dataset_display}__TopSets_Violin_AllIR_VerticalGrid.{FIG_FMT}"
        out_path = os.path.join(OUT_DIR_VIOLIN, fn)
        fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)
        print(f"[INFO] Saved: {out_path}")

# --------- line chart (from unique_feature_lists_df) ---------
def plot_repetition_chart(unique_feature_lists_df: pd.DataFrame, topn: int = 5, save_to_file: bool = False):
    """
    Line chart: Repetition of unique Top-N feature sets across subsets for each IR.
    Uses IR_COLORS (or rank_colors fallback) for consistent coloring.
    """
    if unique_feature_lists_df is None or unique_feature_lists_df.empty:
        print("[WARN] unique_feature_lists_df is empty or not provided.")
        return

    # Determine IR order based on global IR_ORDER (if available in df)
    ir_names = [ir for ir in IR_ORDER if ir in unique_feature_lists_df['imbalanceRatio'].unique()]
    if not ir_names:
        ir_names = unique_feature_lists_df['imbalanceRatio'].unique().tolist()

    x_values = np.arange(1, int(topn) + 1)
    fig, ax = plt.subplots(figsize=(10, 6))

    y_max = 0
    for idx, ir in enumerate(ir_names):
        counts = [(unique_feature_lists_df.query("imbalanceRatio == @ir")['sum'] == x).sum()
                  for x in x_values]
        y_max = max(y_max, max(counts) if counts else 0)

        if USE_IR_COLORS:
            color = IR_COLORS.get(ir, None)
            if color is None:
                # fallback deterministic color if IR not in palette
                color = rank_colors.get((idx % len(rank_colors)) + 1, 'gray')
        else:
            color = rank_colors.get((idx % len(rank_colors)) + 1, 'gray')

        label = IR_LABELS.get(ir, str(ir))
        ax.plot(x_values, counts, marker='o', linestyle='-', label=label, color=color)

    ax.set_xlabel('Number of occurrences')
    ax.set_ylabel('Frequency')
    ax.set_title('Repetition of unique Top-N feature sets across subsets for each IR')
    ax.set_xticks(x_values)

    # Set Y ticks to nice 10-step grid up to max
    if y_max > 0:
        upper = int(math.ceil(y_max / 10.0) * 10)
        ax.set_yticks(np.arange(0, upper + 10, 10))

    ax.legend(title="Imbalance Ratios", loc="upper right")
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

    if save_to_file:
        fn = f"Repetition_UniqueTop{int(topn)}_byIR.{FIG_FMT}"
        out_path = os.path.join(OUT_DIR_VIOLIN, fn)
        fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)
        print(f"[INFO] Saved: {out_path}")

# --------- widgets ---------
w_ds = Dropdown(
    options=[(DISPLAY_NAMES.get(ds, ds), ds) for ds in dataset_identifiers],
    value=dataset_identifiers[0],
    description="Dataset:",
    layout=Layout(width="260px")
)

w_topn = IntSlider(
    value=5, min=1, max=5, step=1,
    description="Top-N:",
    readout=True, readout_format='d',
    continuous_update=False,
    layout=Layout(width="350px")
)

w_unified = Checkbox(value=False, description="Unified axis")
w_save = Checkbox(value=False, description="Save to file")
w_show_rep = Checkbox(value=True, description="Show repetition chart")  # NEW
out = Output()

w_topn.style = {"description_width": "initial"}

def _list_models_for_dataset(ds_value):
    sub = long_df[long_df['dataset'] == ds_value]
    return sorted(sub['method'].dropna().unique().tolist())

def _recompute(dataset, topn, unified_axis, save_to_file, show_rep):
    out.clear_output(wait=True)
    with out:
        # 1) Violin grid 1×4 (all models)
        models = _list_models_for_dataset(dataset)
        if not models:
            print(f"[WARN] No models found for dataset: {dataset}")
            return
        models = models[:4]
        counts_by_model = {m: compute_unique_top_sets(long_df, dataset, topn=topn, model=m) for m in models}
        plot_violin_grid(counts_by_model, dataset, unified_axis=unified_axis, save_to_file=save_to_file)

        # 2) Optional repetition line chart based on unique_feature_lists_df
        if show_rep:
            if 'unique_feature_lists_df' in globals():
                plot_repetition_chart(unique_feature_lists_df, topn=topn, save_to_file=save_to_file)
            else:
                print("[WARN] 'unique_feature_lists_df' is not defined in the notebook scope.")

controls = {
    'dataset': w_ds,
    'topn': w_topn,
    'unified_axis': w_unified,
    'save_to_file': w_save,
    'show_rep': w_show_rep
}

row1 = HBox([w_ds], layout=Layout(align_items="center", gap="16px"))
row2 = HBox([w_topn, w_unified, w_save, w_show_rep], layout=Layout(align_items="center", gap="16px"))
io = interactive_output(_recompute, controls)

display(VBox([row1, row2, out], layout=Layout(gap="8px")))

# initial draw
_recompute(w_ds.value, w_topn.value, w_unified.value, w_save.value, w_show_rep.value)


CPU times: total: 78.1 ms
Wall time: 49.6 ms


# Repetitions of top-N features

In [8]:
# ============================================
# Repetition of unique Top-N feature sets across subsets for each IR
# - full, self-contained Jupyter cell
# - 2×2 grid (up to 4 models), single legend below all subplots
# - independent toggles: "Unified Y axis" and "Unified X axis"
# - skips plotting points where frequency == 0
# - optional saving to a single combined figure (legend included)
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, Layout, interactive_output
from IPython.display import display

# ---------- Required input ----------
# 'long_df' must be present in memory (pd.DataFrame)
try:
    assert isinstance(long_df, pd.DataFrame)
except Exception as _e:
    raise RuntimeError("This cell requires 'long_df' in memory (pd.DataFrame). Define it earlier.") from _e

# ---------- IR order/labels/colors fallbacks ----------
if 'IR_ORDER' not in globals():
    IR_ORDER = sorted(long_df['IR'].dropna().unique().tolist(), key=lambda x: (str(type(x)), x))
if 'IR_LABELS' not in globals():
    IR_LABELS = {ir: str(ir) for ir in IR_ORDER}
if 'IR_COLORS' not in globals():
    cmap = plt.get_cmap('tab20')
    IR_COLORS = {ir: cmap(i % 20) for i, ir in enumerate(IR_ORDER)}

# ---------- Display name map / figure params / base path fallbacks ----------
if 'DISPLAY_NAMES' not in globals():
    DISPLAY_NAMES = {}
if 'FIG_FMT' not in globals():
    FIG_FMT = 'png'
if 'FIG_DPI' not in globals():
    FIG_DPI = 160
if 'base_path' not in globals():
    base_path = '.'

# ---------- Dataset identifiers for the selector ----------
if 'dataset_identifiers' not in globals():
    dataset_identifiers = sorted(long_df['dataset'].dropna().unique().tolist())

# ---------- Output directory ----------
OUT_DIR_REP_GRID = os.path.join(base_path, "figs_TOPsets_repetition_lines_byIR_GRID2x2")
os.makedirs(OUT_DIR_REP_GRID, exist_ok=True)

# ---------- Core logic: compute unique Top-N sets and their repetitions ----------
def _clamp_topn(n: int, lo: int = 1, hi: int = 10) -> int:
    try:
        n = int(n)
    except Exception:
        n = lo
    return int(max(lo, min(hi, n)))

def compute_unique_top_sets__DUP(df: pd.DataFrame, dataset: str, topn: int = 5, model: str | None = None) -> pd.DataFrame:
    """
    For a selected dataset (and optional single model), compute unique Top-N feature sets
    across (method × subset). Returns:
        [IR, IR_label, top_set (tuple(feature)), count]
    where 'count' is the number of repetitions of an identical Top-N set.
    """
    g = df[df['dataset'] == dataset].copy()
    if model is not None:
        g = g[g['method'] == model]
    if g.empty:
        return pd.DataFrame(columns=["IR", "IR_label", "top_set", "count"])

    # Ensure no NaN in 'subset'
    g['subset_filled'] = g['subset'].fillna(-1)

    # Clamp Top-N
    topn = _clamp_topn(topn, lo=1, hi=10)

    recs = []
    # Group by (method, subset, IR) — each bin defines one Top-N selection
    for (_, subset_val, ir), sub in g.groupby(['method', 'subset_filled', 'IR'], dropna=False):
        # Mean absolute importance per feature
        imp = (sub.assign(a=sub['shapMean'].abs())
                   .groupby('feature', as_index=False)['a'].mean()
                   .sort_values('a', ascending=False))
        k = min(topn, len(imp))
        if k == 0:
            continue
        top_feats = tuple(imp['feature'].head(k))
        recs.append({"IR": ir, "top_set": top_feats})

    if not recs:
        return pd.DataFrame(columns=["IR", "IR_label", "top_set", "count"])

    # Count repetitions of identical Top-N sets per IR
    counts = (pd.DataFrame(recs)
                .groupby(['IR', 'top_set'])
                .size()
                .reset_index(name='count'))
    counts['IR_label'] = counts['IR'].map(IR_LABELS)
    counts['IR_order_idx'] = counts['IR'].apply(lambda x: IR_ORDER.index(x) if x in IR_ORDER else 999)
    counts = counts.sort_values(['IR_order_idx', 'count'], ascending=[True, False]).drop(columns='IR_order_idx')
    return counts

def compute_repetition_distribution(counts_df: pd.DataFrame) -> pd.DataFrame:
    """
    From [IR, top_set, count], build a distribution per IR:
      x_occurrences = number of repetitions (x),
      frequency = how many unique Top-N sets occur exactly x times.
    IMPORTANT: rows with frequency == 0 are removed (not plotted).
    Returns: [IR, IR_label, x_occurrences, frequency] with frequency > 0.
    """
    if counts_df.empty:
        return pd.DataFrame(columns=["IR","IR_label","x_occurrences","frequency"])

    out = []
    for ir, sub in counts_df.groupby("IR"):
        max_c = int(sub["count"].max())
        for x in range(1, max_c + 1):
            freq = int((sub["count"] == x).sum())
            if freq > 0:  # drop y=0 entries
                out.append({
                    "IR": ir,
                    "IR_label": IR_LABELS.get(ir, str(ir)),
                    "x_occurrences": x,
                    "frequency": freq
                })

    if not out:
        return pd.DataFrame(columns=["IR","IR_label","x_occurrences","frequency"])

    dist = pd.DataFrame(out)
    dist["IR_order_idx"] = dist["IR"].apply(lambda r: IR_ORDER.index(r) if r in IR_ORDER else 999)
    return dist.sort_values(["IR_order_idx","x_occurrences"]).drop(columns="IR_order_idx")

# ---------- Plotting (2×2 grid, one subplot per model) ----------
def plot_repetition_lines_grid_2x2(df: pd.DataFrame, dataset: str, topn: int,
                                   unified_y: bool = False, unified_x: bool = False,
                                   save_to_file: bool = False):
    """
    Builds a 2×2 grid (up to 4 models), each subplot shows curves for all IRs
    for a single model. If unified_y is True, all subplots share the same Y limits/ticks.
    If unified_x is True, all subplots share the same X limits/ticks.
    """
    ds_df = df[df['dataset'] == dataset]
    if ds_df.empty:
        ds_name = DISPLAY_NAMES.get(dataset, dataset)
        print(f"[WARN] No data for dataset: {ds_name}")
        return

    models_all = sorted(ds_df['method'].dropna().unique().tolist())
    if not models_all:
        print("[WARN] No models found for the selected dataset.")
        return

    models = models_all[:4]  # limit to 4 for 2×2 grid
    if len(models_all) > 4:
        print(f"[INFO] More than 4 models detected ({len(models_all)}). Showing first 4: {models}")

    # Precompute distributions per model and collect global maxima if needed
    model_to_dist = {}
    global_y_max = 0
    global_x_max = 0
    for m in models:
        counts = compute_unique_top_sets__DUP(df, dataset, topn=topn, model=m)
        dist   = compute_repetition_distribution(counts)
        model_to_dist[m] = dist
        if not dist.empty:
            global_y_max = max(global_y_max, int(dist['frequency'].max()))
            global_x_max = max(global_x_max, int(dist['x_occurrences'].max()))

    # Prepare figure geometry; sharex/sharey based on toggles
    n_rows, n_cols = 2, 2
    n_plots = n_rows * n_cols
    fig_width = 12
    fig_height = 8
    fig, axes = plt.subplots(
        nrows=n_rows, ncols=n_cols,
        figsize=(fig_width, fig_height),
        sharex=unified_x, sharey=unified_y
    )
    axes = np.array(axes).reshape(-1)  # flatten to length 4

    dataset_display = DISPLAY_NAMES.get(dataset, dataset)
    fig.suptitle(
        f"{dataset_display} — Repetition of unique Top-{topn} feature sets (up to 4 models)",
        y=0.98, fontsize=12
    )

    # Determine unified ticks/limits if requested
    if unified_y:
        if global_y_max <= 0:
            global_y_max = 1
        step_y = max(1, int(np.ceil(global_y_max / 6)))  # 6 grid lines approx
        y_ticks = np.arange(1, global_y_max + 1, step_y)

    if unified_x:
        if global_x_max <= 0:
            global_x_max = 1
        x_ticks = np.arange(1, global_x_max + 1, 1)

    # Draw each subplot; collect legend handles/labels once
    fig_handles, fig_labels = [], []
    for ax, m in zip(axes, models):
        dist = model_to_dist[m]
        if dist.empty:
            ax.text(0.5, 0.5, "No data", ha='center', va='center')
            ax.set_title(f"Model: {m}")
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            continue

        present_irs = [ir for ir in IR_ORDER if ir in dist["IR"].unique()] or dist["IR"].unique().tolist()
        # Plot one curve per IR
        local_handles = []
        local_labels  = []
        for ir in present_irs:
            sub = dist[dist["IR"] == ir]
            if sub.empty:
                continue
            xs = sub["x_occurrences"].to_numpy()
            ys = sub["frequency"].to_numpy()
            label = IR_LABELS.get(ir, str(ir))
            color = IR_COLORS.get(ir, None)
            line, = ax.plot(xs, ys, marker='o', linestyle='-', label=label, color=color)
            local_handles.append(line); local_labels.append(label)

        # Merge legend items (preserve order, avoid duplicates)
        for h, l in zip(local_handles, local_labels):
            if l not in fig_labels:
                fig_handles.append(h)
                fig_labels.append(l)

        # Axes cosmetics
        ax.set_title(f"Model: {m}")
        ax.set_xlabel("Number of occurrences (across method × subset)")
        ax.set_ylabel("Frequency (unique Top-N sets)")

        if unified_x:
            ax.set_xticks(x_ticks)
            ax.set_xlim(left=1, right=max(1, global_x_max))
        else:
            unique_xs = sorted(dist["x_occurrences"].unique().tolist())
            ax.set_xticks(unique_xs)
            if unique_xs:
                ax.set_xlim(left=min(unique_xs), right=max(unique_xs))

        if unified_y:
            ax.set_yticks(y_ticks)
            ax.set_ylim(bottom=1, top=max(1, global_y_max))
        else:
            local_max = int(dist["frequency"].max())
            if local_max <= 0:
                local_max = 1
            step_y_local = max(1, int(np.ceil(local_max / 6)))
            ax.set_yticks(np.arange(1, local_max + 1, step_y_local))
            ax.set_ylim(bottom=1, top=local_max)

        ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Hide unused axes if fewer than 4 models
    for ax in axes[len(models):]:
        ax.axis('off')

    # Single legend at the bottom
    if fig_labels:
        ncol = min(len(fig_labels), 6)
        fig.legend(
            handles=fig_handles, labels=fig_labels,
            loc='lower center', ncol=ncol,
            bbox_to_anchor=(0.5, -0.02), frameon=False
        )
        plt.subplots_adjust(bottom=0.12, top=0.92)
    else:
        plt.subplots_adjust(top=0.92)

    plt.tight_layout(rect=[0, 0.06, 1, 0.92])  # leave space for legend & title
    plt.show()

    if save_to_file:
        fn = f"{dataset_display}__UP_TO4_MODELS__Top{topn}_RepetitionLines_GRID2x2" \
             f"{'_unifiedY' if unified_y else ''}{'_unifiedX' if unified_x else ''}.{FIG_FMT}"
        out_path = os.path.join(OUT_DIR_REP_GRID, fn)

        # Rebuild the figure for a clean save (same logic)
        fig, axes = plt.subplots(
            nrows=2, ncols=2,
            figsize=(fig_width, fig_height),
            sharex=unified_x, sharey=unified_y
        )
        axes = np.array(axes).reshape(-1)
        fig.suptitle(
            f"{dataset_display} — Repetition of unique Top-{topn} feature sets (up to 4 models)",
            y=0.98, fontsize=12
        )

        if unified_y:
            if global_y_max <= 0:
                global_y_max = 1
            step_y = max(1, int(np.ceil(global_y_max / 6)))
            y_ticks = np.arange(1, global_y_max + 1, step_y)
        if unified_x:
            if global_x_max <= 0:
                global_x_max = 1
            x_ticks = np.arange(1, global_x_max + 1, 1)

        fig_handles, fig_labels = [], []
        for ax, m in zip(axes, models):
            dist = model_to_dist[m]
            if dist.empty:
                ax.text(0.5, 0.5, "No data", ha='center', va='center')
                ax.set_title(f"Model: {m}")
                ax.grid(axis='y', linestyle='--', alpha=0.7)
                continue

            present_irs = [ir for ir in IR_ORDER if ir in dist["IR"].unique()] or dist["IR"].unique().tolist()
            local_handles = []; local_labels = []
            for ir in present_irs:
                sub = dist[dist["IR"] == ir]
                if sub.empty:
                    continue
                xs = sub["x_occurrences"].to_numpy()
                ys = sub["frequency"].to_numpy()
                label = IR_LABELS.get(ir, str(ir))
                color = IR_COLORS.get(ir, None)
                line, = ax.plot(xs, ys, marker='o', linestyle='-', label=label, color=color)
                local_handles.append(line); local_labels.append(label)

            for h, l in zip(local_handles, local_labels):
                if l not in fig_labels:
                    fig_handles.append(h)
                    fig_labels.append(l)

            ax.set_title(f"Model: {m}")
            ax.set_xlabel("Number of occurrences (across method × subset)")
            ax.set_ylabel("Frequency (unique Top-N sets)")

            if unified_x:
                ax.set_xticks(x_ticks)
                ax.set_xlim(left=1, right=max(1, global_x_max))
            else:
                unique_xs = sorted(dist["x_occurrences"].unique().tolist())
                ax.set_xticks(unique_xs)
                if unique_xs:
                    ax.set_xlim(left=min(unique_xs), right=max(unique_xs))

            if unified_y:
                ax.set_yticks(y_ticks)
                ax.set_ylim(bottom=1, top=max(1, global_y_max))
            else:
                local_max = int(dist["frequency"].max())
                if local_max <= 0:
                    local_max = 1
                step_y_local = max(1, int(np.ceil(local_max / 6)))
                ax.set_yticks(np.arange(1, local_max + 1, step_y_local))
                ax.set_ylim(bottom=1, top=local_max)

            ax.grid(axis='y', linestyle='--', alpha=0.7)

        for ax in axes[len(models):]:
            ax.axis('off')

        if fig_labels:
            ncol = min(len(fig_labels), 6)
            fig.legend(
                handles=fig_handles, labels=fig_labels,
                loc='lower center', ncol=ncol,
                bbox_to_anchor=(0.5, -0.02), frameon=False
            )
            plt.subplots_adjust(bottom=0.12, top=0.92)
        else:
            plt.subplots_adjust(top=0.92)

        plt.tight_layout(rect=[0, 0.06, 1, 0.92])
        plt.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.close()
        print(f"[INFO] Saved: {out_path}")

# ---------- Widgets (dataset + controls; no model selector because we show up to 4) ----------
w_dsG = Dropdown(
    options=[(DISPLAY_NAMES.get(ds, ds), ds) for ds in dataset_identifiers],
    value=dataset_identifiers[0] if dataset_identifiers else None,
    description="Dataset:",
    layout=Layout(width="260px")
)

w_topnG = IntSlider(
    value=5, min=1, max=10, step=1,
    description="Top-N:",
    readout=True, readout_format='d',
    continuous_update=False,
    layout=Layout(width="350px")
)

w_unifiedYG = Checkbox(value=False, description="Unified Y axis")
w_unifiedXG = Checkbox(value=False, description="Unified X axis")
w_saveG     = Checkbox(value=False, description="Save to file")
outG = Output()

# Nicer label alignment
w_topnG.style = {"description_width": "initial"}

def _recompute_grid(dataset, topn, unified_y, unified_x, save_to_file):
    outG.clear_output(wait=True)
    with outG:
        plot_repetition_lines_grid_2x2(
            long_df, dataset, topn,
            unified_y=unified_y, unified_x=unified_x,
            save_to_file=save_to_file
        )

controlsG = {
    'dataset': w_dsG,
    'topn': w_topnG,
    'unified_y': w_unifiedYG,
    'unified_x': w_unifiedXG,
    'save_to_file': w_saveG
}

row1g = HBox([w_dsG], layout=Layout(align_items="center", gap="16px"))
row2g = HBox([w_topnG, w_unifiedYG, w_unifiedXG, w_saveG], layout=Layout(align_items="center", gap="16px"))
ioG = interactive_output(_recompute_grid, controlsG)

display(VBox([row1g, row2g, outG], layout=Layout(gap="8px")))

# Initial draw
if dataset_identifiers:
    _recompute_grid(w_dsG.value, w_topnG.value, w_unifiedYG.value, w_unifiedXG.value, w_saveG.value)
else:
    print("[WARN] No available datasets in 'dataset_identifiers'.")


# Feature set entropy across IRs

In [13]:
# ============================================
# Feature Set Entropy of Top-N feature sets across subsets for each IR
# - full, self-contained Jupyter cell
# - shows ALL models simultaneously in a 1×N grid (one column, N rows)
# - independent toggles: "Unified Y axis" and "Unified X axis"
# - optional saving to a single combined figure
# - uses the same globals (IR_ORDER/LABELS/COLORS/DISPLAY_NAMES/FIG_FMT/FIG_DPI/base_path)
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, Layout, interactive_output
from IPython.display import display
from scipy.stats import entropy as shannon_entropy

# ---------- Required input ----------
try:
    assert isinstance(long_df, pd.DataFrame)
except Exception as _e:
    raise RuntimeError("This cell requires 'long_df' in memory (pd.DataFrame). Define it earlier.") from _e

# ---------- IR order/labels/colors fallbacks ----------
if 'IR_ORDER' not in globals():
    IR_ORDER = sorted(long_df['IR'].dropna().unique().tolist(), key=lambda x: (str(type(x)), x))
if 'IR_LABELS' not in globals():
    IR_LABELS = {ir: str(ir) for ir in IR_ORDER}
if 'IR_COLORS' not in globals():
    cmap = plt.get_cmap('tab20')
    IR_COLORS = {ir: cmap(i % 20) for i, ir in enumerate(IR_ORDER)}

# ---------- Display name / figure params / base path fallbacks ----------
if 'DISPLAY_NAMES' not in globals():
    DISPLAY_NAMES = {}
if 'FIG_FMT' not in globals():
    FIG_FMT = 'png'
if 'FIG_DPI' not in globals():
    FIG_DPI = 160
if 'base_path' not in globals():
    base_path = '.'

# ---------- Dataset identifiers ----------
if 'dataset_identifiers' not in globals():
    dataset_identifiers = sorted(long_df['dataset'].dropna().unique().tolist())

# ---------- Output directory ----------
OUT_DIR_ENT_GRID = os.path.join(base_path, "figs_TOPsets_entropy_byIR_GRID1xN")
os.makedirs(OUT_DIR_ENT_GRID, exist_ok=True)

# ---------- Utils ----------
def _clamp(n: int, lo: int = 1, hi: int = 10) -> int:
    try:
        n = int(n)
    except Exception:
        n = lo
    return int(max(lo, min(hi, n)))

# ---------- Build ranked feature lists per (subset × IR) for a given model ----------
def compute_ranked_features_per_bin(df: pd.DataFrame, dataset: str, model: str) -> pd.DataFrame:
    """
    For a selected dataset and model, compute the full ranking of features (by mean abs shapMean)
    within each (subset, IR). Returns DataFrame:
        [IR, IR_label, subset_filled, ranked_features (tuple in desc. importance)]
    """
    g = df[(df['dataset'] == dataset) & (df['method'] == model)].copy()
    if g.empty:
        return pd.DataFrame(columns=["IR", "IR_label", "subset_filled", "ranked_features"])

    g['subset_filled'] = g['subset'].fillna(-1)

    recs = []
    for (subset_val, ir), sub in g.groupby(['subset_filled', 'IR'], dropna=False):
        imp = (sub.assign(a=sub['shapMean'].abs())
                 .groupby('feature', as_index=False)['a'].mean()
                 .sort_values('a', ascending=False))
        ranked = tuple(imp['feature'].tolist())
        if len(ranked) == 0:
            continue
        recs.append({
            "IR": ir,
            "IR_label": IR_LABELS.get(ir, str(ir)),
            "subset_filled": subset_val,
            "ranked_features": ranked
        })

    out = pd.DataFrame(recs)
    if out.empty:
        return out

    out["IR_order_idx"] = out["IR"].apply(lambda r: IR_ORDER.index(r) if r in IR_ORDER else 999)
    return out.sort_values(["IR_order_idx", "subset_filled"]).drop(columns="IR_order_idx")

# ---------- Entropy for Top-k feature sets ----------
def feature_set_entropy_for_k(feature_orders: list[tuple], k: int) -> float:
    """
    Given a list of full rankings (tuples) for multiple subsets (same IR),
    compute Shannon entropy of the distribution of Top-k sets.
    Steps:
      - form Top-k tuple for each subset
      - count identical tuples
      - normalize to probabilities
      - entropy(prob_dist) with natural logarithm
    Edge cases:
      - 0 or 1 subset -> entropy = 0.0
    """
    if not feature_orders or len(feature_orders) <= 1:
        return 0.0

    topk_tuples = [tuple(r[:k]) for r in feature_orders]
    _, counts = np.unique(topk_tuples, return_counts=True, axis=0)
    p = counts / counts.sum()
    return float(shannon_entropy(p))  # natural log base

def compute_entropy_curves(df: pd.DataFrame, dataset: str, model: str, k_max: int) -> pd.DataFrame:
    """
    For given dataset and model, returns tidy DF with columns:
        [IR, IR_label, k, entropy]
    where k runs 1..k_max.
    """
    k_max = _clamp(k_max, lo=1, hi=10)

    ranks_df = compute_ranked_features_per_bin(df, dataset, model)
    if ranks_df.empty:
        return pd.DataFrame(columns=["IR", "IR_label", "k", "entropy"])

    out = []
    for ir, sub in ranks_df.groupby("IR"):
        orders = sub["ranked_features"].tolist()
        for k in range(1, k_max + 1):
            ent = feature_set_entropy_for_k(orders, k)
            out.append({
                "IR": ir,
                "IR_label": IR_LABELS.get(ir, str(ir)),
                "k": k,
                "entropy": ent
            })
    res = pd.DataFrame(out)
    res["IR_order_idx"] = res["IR"].apply(lambda r: IR_ORDER.index(r) if r in IR_ORDER else 999)
    return res.sort_values(["IR_order_idx", "k"]).drop(columns="IR_order_idx")

# ---------- Plotting (1×N grid, one subplot per model) ----------
def plot_entropy_lines_grid(df: pd.DataFrame, dataset: str, k_max: int,
                            unified_y: bool = False, unified_x: bool = False,
                            save_to_file: bool = False):
    """
    Builds a 1×N grid (N rows, 1 column) where each subplot shows entropy curves (IR lines)
    vs Top-N for a single model.
    """
    ds_df = df[df['dataset'] == dataset]
    if ds_df.empty:
        ds_name = DISPLAY_NAMES.get(dataset, dataset)
        print(f"[WARN] No data for dataset: {ds_name}")
        return

    models = sorted(ds_df['method'].dropna().unique().tolist())
    if not models:
        print("[WARN] No models found for the selected dataset.")
        return

    # Precompute curves and global limits if needed
    model_to_curves = {}
    global_y_min, global_y_max = np.inf, -np.inf
    global_x_max = 0
    for m in models:
        curves = compute_entropy_curves(df, dataset, m, k_max)
        model_to_curves[m] = curves
        if not curves.empty:
            ymax = float(np.nanmax(curves['entropy'].to_numpy()))
            ymin = float(np.nanmin(curves['entropy'].to_numpy()))
            global_y_max = max(global_y_max, ymax)
            global_y_min = min(global_y_min, ymin)
            global_x_max = max(global_x_max, int(curves['k'].max()))

    # Fallbacks if all empty
    if not np.isfinite(global_y_min):
        global_y_min = 0.0
    if not np.isfinite(global_y_max):
        global_y_max = 1.0

    # Prepare figure
    n_models = len(models)
    fig_height = max(4.0 * n_models, 4.0)
    fig, axes = plt.subplots(
        nrows=n_models, ncols=1,
        figsize=(10, fig_height),
        sharex=unified_x, sharey=unified_y
    )
    if n_models == 1:
        axes = [axes]

    dataset_display = DISPLAY_NAMES.get(dataset, dataset)
    fig.suptitle(
        f"{dataset_display} — Feature set entropy of Top-N sets (all models)",
        y=0.995, fontsize=12
    )

    # Unified ticks/limits
    if unified_x:
        if global_x_max <= 0:
            global_x_max = _clamp(k_max, lo=1, hi=10)
        x_ticks = np.arange(1, global_x_max + 1, 1)

    if unified_y:
        # Entropia jest >= 0; górna granica zależy od liczby unikalnych zestawów
        # Przyjmijmy zakres [0, ceil(global_y_max*10)/10] z drobną poduszką
        y_lo = 0.0
        y_hi = float(np.ceil(max(global_y_max, 0.01) * 10.0) / 10.0)
        y_ticks = np.linspace(y_lo, y_hi, num=6)

    # Draw per-model subplot
    for ax, m in zip(axes, models):
        curves = model_to_curves[m]
        if curves.empty:
            ax.text(0.5, 0.5, "No data", ha='center', va='center')
            ax.set_title(f"Model: {m}")
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            continue

        present_irs = [ir for ir in IR_ORDER if ir in curves["IR"].unique()] or curves["IR"].unique().tolist()
        for ir in present_irs:
            sub = curves[curves["IR"] == ir]
            if sub.empty:
                continue
            xs = sub["k"].to_numpy()
            ys = sub["entropy"].to_numpy(dtype=float)
            label = IR_LABELS.get(ir, str(ir))
            color = IR_COLORS.get(ir, None)
            ax.plot(xs, ys, marker='o', linestyle='-', label=label, color=color)

        ax.set_title(f"Model: {m}")
        ax.set_xlabel("Top-N Features (k)")
        ax.set_ylabel("Feature set entropy (nats)")

        if unified_x:
            ax.set_xticks(x_ticks)
            ax.set_xlim(left=1, right=max(1, global_x_max))
        else:
            unique_xs = sorted(curves["k"].unique().tolist())
            ax.set_xticks(unique_xs)
            if unique_xs:
                ax.set_xlim(left=min(unique_xs), right=max(unique_xs))

        if unified_y:
            ax.set_yticks(y_ticks)
            ax.set_ylim(bottom=y_ticks[0], top=y_ticks[-1])
        else:
            y_min = float(np.nanmin(curves["entropy"].to_numpy()))
            y_max = float(np.nanmax(curves["entropy"].to_numpy()))
            if not np.isfinite(y_min) or not np.isfinite(y_max):
                y_min, y_max = 0.0, 1.0
            pad = max(1e-3, (y_max - y_min) * 0.1)
            ax.set_ylim(bottom=max(0.0, y_min - pad), top=y_max + pad)

        ax.legend(title="Imbalance Ratios", loc="upper left")
        ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

    # Optional save-to-file (clean rebuild)
    if save_to_file:
        fn = f"{dataset_display}__ALL_MODELS__Entropy_GRID1x{n_models}" \
             f"{'_unifiedY' if unified_y else ''}{'_unifiedX' if unified_x else ''}.{FIG_FMT}"
        out_path = os.path.join(OUT_DIR_ENT_GRID, fn)

        fig, axes = plt.subplots(
            nrows=n_models, ncols=1,
            figsize=(10, fig_height),
            sharex=unified_x, sharey=unified_y
        )
        if n_models == 1:
            axes = [axes]
        fig.suptitle(
            f"{dataset_display} — Feature set entropy of Top-N sets (all models)",
            y=0.995, fontsize=12
        )

        if unified_x:
            if global_x_max <= 0:
                global_x_max = _clamp(k_max, lo=1, hi=10)
            x_ticks = np.arange(1, global_x_max + 1, 1)
        if unified_y:
            y_lo = 0.0
            y_hi = float(np.ceil(max(global_y_max, 0.01) * 10.0) / 10.0)
            y_ticks = np.linspace(y_lo, y_hi, num=6)

        for ax, m in zip(axes, models):
            curves = model_to_curves[m]
            if curves.empty:
                ax.text(0.5, 0.5, "No data", ha='center', va='center')
                ax.set_title(f"Model: {m}")
                ax.grid(axis='y', linestyle='--', alpha=0.7)
                continue

            present_irs = [ir for ir in IR_ORDER if ir in curves["IR"].unique()] or curves["IR"].unique().tolist()
            for ir in present_irs:
                sub = curves[curves["IR"] == ir]
                xs = sub["k"].to_numpy()
                ys = sub["entropy"].to_numpy(dtype=float)
                label = IR_LABELS.get(ir, str(ir))
                color = IR_COLORS.get(ir, None)
                ax.plot(xs, ys, marker='o', linestyle='-', label=label, color=color)

            ax.set_title(f"Model: {m}")
            ax.set_xlabel("Top-N Features (k)")
            ax.set_ylabel("Feature set entropy (nats)")

            if unified_x:
                ax.set_xticks(x_ticks)
                ax.set_xlim(left=1, right=max(1, global_x_max))
            else:
                unique_xs = sorted(curves["k"].unique().tolist())
                ax.set_xticks(unique_xs)
                if unique_xs:
                    ax.set_xlim(left=min(unique_xs), right=max(unique_xs))

            if unified_y:
                ax.set_yticks(y_ticks)
                ax.set_ylim(bottom=y_ticks[0], top=y_ticks[-1])
            else:
                y_min = float(np.nanmin(curves["entropy"].to_numpy()))
                y_max = float(np.nanmax(curves["entropy"].to_numpy()))
                if not np.isfinite(y_min) or not np.isfinite(y_max):
                    y_min, y_max = 0.0, 1.0
                pad = max(1e-3, (y_max - y_min) * 0.1)
                ax.set_ylim(bottom=max(0.0, y_min - pad), top=y_max + pad)

            ax.legend(title="Imbalance Ratios", loc="upper left")
            ax.grid(axis='y', linestyle='--', alpha=0.7)

        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.close()
        print(f"[INFO] Saved: {out_path}")

# ---------- Widgets (dataset + controls; no model selector because we show all) ----------
w_dsE = Dropdown(
    options=[(DISPLAY_NAMES.get(ds, ds), ds) for ds in dataset_identifiers],
    value=dataset_identifiers[0] if dataset_identifiers else None,
    description="Dataset:",
    layout=Layout(width="260px")
)

w_kmaxE = IntSlider(
    value=5, min=1, max=10, step=1,
    description="Top-N (max):",
    readout=True, readout_format='d',
    continuous_update=False,
    layout=Layout(width="350px")
)

w_unifiedYE = Checkbox(value=False, description="Unified Y axis")
w_unifiedXE = Checkbox(value=False, description="Unified X axis")
w_saveE     = Checkbox(value=False, description="Save to file")
outE = Output()

# Align labels nicely
w_kmaxE.style = {"description_width": "initial"}

def _recompute_entropy_grid(dataset, k_max, unified_y, unified_x, save_to_file):
    outE.clear_output(wait=True)
    with outE:
        plot_entropy_lines_grid(
            long_df, dataset, k_max,
            unified_y=unified_y, unified_x=unified_x,
            save_to_file=save_to_file
        )

controlsE = {
    'dataset': w_dsE,
    'k_max': w_kmaxE,
    'unified_y': w_unifiedYE,
    'unified_x': w_unifiedXE,
    'save_to_file': w_saveE
}

row1e = HBox([w_dsE], layout=Layout(align_items="center", gap="16px"))
row2e = HBox([w_kmaxE, w_unifiedYE, w_unifiedXE, w_saveE], layout=Layout(align_items="center", gap="16px"))
ioE = interactive_output(_recompute_entropy_grid, controlsE)

display(VBox([row1e, row2e, outE], layout=Layout(gap="8px")))

# Initial draw
if dataset_identifiers:
    _recompute_entropy_grid(w_dsE.value, w_kmaxE.value, w_unifiedYE.value, w_unifiedXE.value, w_saveE.value)
else:
    print("[WARN] No available datasets in 'dataset_identifiers'.")


In [9]:
# ============================================
# Violin plots: Feature set entropy across IRs (all models; 1×N vertical)
# - uses long_df with columns: ['dataset','method','subset','IR','feature','shapMean']
# - one subplot per model; each shows violins (po jednym na IR)
# - dane do skrzypiec: entropia rozkładu unikalnych Top-k zestawów dla k=1..K (w obrębie IR)
# - widgets: Dataset, Top-N (max K), Unified Y axis, Save to file
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, Layout, interactive_output
from IPython.display import display
from scipy.stats import entropy as shannon_entropy
from matplotlib.patches import Patch

# ---------- Required input ----------
try:
    assert isinstance(long_df, pd.DataFrame)
except Exception as _e:
    raise RuntimeError("This cell requires 'long_df' in memory (pd.DataFrame) with columns "
                       "['dataset','method','subset','IR','feature','shapMean'].") from _e

_required_cols = {'dataset','method','subset','IR','feature','shapMean'}
missing_cols = _required_cols - set(long_df.columns)
if missing_cols:
    raise RuntimeError(f"'long_df' is missing required columns: {sorted(missing_cols)}")

# ---------- Fallbacks ----------
if 'IR_ORDER' not in globals():
    IR_ORDER = sorted(long_df['IR'].dropna().unique().tolist(), key=lambda x: (str(type(x)), x))
if 'IR_LABELS' not in globals():
    IR_LABELS = {ir: str(ir) for ir in IR_ORDER}
if 'IR_COLORS' not in globals():
    cmap = plt.get_cmap('tab20')
    IR_COLORS = {ir: cmap(i % 20) for i, ir in enumerate(IR_ORDER)}
if 'DISPLAY_NAMES' not in globals():
    DISPLAY_NAMES = {}
if 'FIG_FMT' not in globals():
    FIG_FMT = 'png'
if 'FIG_DPI' not in globals():
    FIG_DPI = 160
if 'base_path' not in globals():
    base_path = '.'
if 'dataset_identifiers' not in globals():
    dataset_identifiers = sorted(long_df['dataset'].dropna().unique().tolist())

# ---------- Output directory ----------
OUT_DIR_VIOLIN_ENT = os.path.join(base_path, "figs_Entropy_violin_byIR_GRID1xN")
os.makedirs(OUT_DIR_VIOLIN_ENT, exist_ok=True)

# ---------- Utils ----------
def _clamp(n: int, lo: int = 1, hi: int = 10) -> int:
    try:
        n = int(n)
    except Exception:
        n = lo
    return int(max(lo, min(hi, n)))

# ---------- Build ranked feature lists per (subset × IR) for a given model ----------
def compute_ranked_features_per_bin(df: pd.DataFrame, dataset: str, model: str) -> pd.DataFrame:
    """
    For a selected dataset and model, compute the full ranking of features (by mean abs shapMean)
    within each (subset, IR). Returns:
        [IR, IR_label, subset_filled, ranked_features (tuple in desc. importance)]
    """
    g = df[(df['dataset'] == dataset) & (df['method'] == model)].copy()
    if g.empty:
        return pd.DataFrame(columns=["IR", "IR_label", "subset_filled", "ranked_features"])

    g['subset_filled'] = g['subset'].fillna(-1)

    recs = []
    for (subset_val, ir), sub in g.groupby(['subset_filled', 'IR'], dropna=False):
        imp = (sub.assign(a=sub['shapMean'].abs())
                 .groupby('feature', as_index=False)['a'].mean()
                 .sort_values('a', ascending=False))
        ranked = tuple(imp['feature'].tolist())
        if ranked:
            recs.append({
                "IR": ir,
                "IR_label": IR_LABELS.get(ir, str(ir)),
                "subset_filled": subset_val,
                "ranked_features": ranked
            })

    out = pd.DataFrame(recs)
    if out.empty:
        return out

    out["IR_order_idx"] = out["IR"].apply(lambda r: IR_ORDER.index(r) if r in IR_ORDER else 999)
    return out.sort_values(["IR_order_idx", "subset_filled"]).drop(columns="IR_order_idx")

# ---------- Entropy for Top-k feature sets ----------
def feature_set_entropy_for_k(feature_orders, k: int) -> float:
    """
    feature_orders: list[tuple] pełnych rankingów dla subsetów w obrębie jednego IR.
    Kroki:
      - topk_tuples: Top-k jako krotki,
      - policz identyczne krotki,
      - p = counts/sum,
      - entropia Shannona (log naturalny).
    Edge cases:
      - 0 lub 1 subset -> 0.0
    """
    if not feature_orders or len(feature_orders) <= 1:
        return 0.0
    topk_tuples = [tuple(r[:k]) for r in feature_orders]
    _, counts = np.unique(topk_tuples, return_counts=True, axis=0)
    p = counts / counts.sum()
    return float(shannon_entropy(p))  # nats

def compute_entropy_by_IR_over_k(df: pd.DataFrame, dataset: str, model: str, k_max: int) -> pd.DataFrame:
    """
    Zwraca tidy DF: [IR, IR_label, k, entropy] dla k=1..k_max.
    """
    k_max = _clamp(k_max, lo=1, hi=10)
    ranks = compute_ranked_features_per_bin(df, dataset, model)
    if ranks.empty:
        return pd.DataFrame(columns=["IR", "IR_label", "k", "entropy"])

    recs = []
    for ir, sub in ranks.groupby("IR"):
        orders = sub["ranked_features"].tolist()
        for k in range(1, k_max + 1):
            recs.append({
                "IR": ir,
                "IR_label": IR_LABELS.get(ir, str(ir)),
                "k": k,
                "entropy": feature_set_entropy_for_k(orders, k)
            })
    res = pd.DataFrame(recs)
    res["IR_order_idx"] = res["IR"].apply(lambda r: IR_ORDER.index(r) if r in IR_ORDER else 999)
    return res.sort_values(["IR_order_idx", "k"]).drop(columns="IR_order_idx")

# ---------- Helpers for violin ----------
def _collect_violin_data_from_entropy(curves_df: pd.DataFrame):
    """
    Zwraca: data_by_ir (lista 1D tablic y), tick_labels, body_colors, present_irs
    Każda tablica y to wartości entropii dla kolejnych k w jednym IR.
    """
    if curves_df is None or curves_df.empty:
        return [], [], [], []

    present_irs = [ir for ir in IR_ORDER if ir in curves_df["IR"].unique()]
    if not present_irs:
        present_irs = curves_df["IR"].unique().tolist()

    data_by_ir, tick_labels, body_colors = [], [], []
    for ir in present_irs:
        sub = curves_df[curves_df["IR"] == ir].sort_values("k")
        ys = sub["entropy"].to_numpy(dtype=float)
        if ys.size == 0:
            continue
        data_by_ir.append(ys)
        tick_labels.append(IR_LABELS.get(ir, str(ir)))
        body_colors.append(IR_COLORS.get(ir, None))
    return data_by_ir, tick_labels, body_colors, present_irs

def _draw_single_violin(ax, data_by_ir, tick_labels, body_colors, title_text):
    parts = ax.violinplot(
        dataset=data_by_ir,
        showmeans=True,
        showmedians=True,
        showextrema=True
    )
    # kolorowanie korpusów wg IR_COLORS
    for i, b in enumerate(parts['bodies']):
        col = body_colors[i]
        if col is not None:
            b.set_facecolor(col)
        b.set_alpha(0.7)
        b.set_edgecolor('black')
        b.set_linewidth(0.8)
    for k in ('cmeans', 'cmedians', 'cbars', 'cmins', 'cmaxes'):
        if k in parts:
            parts[k].set_linewidth(1.2)
            parts[k].set_color('black')

    ax.set_xticks(np.arange(1, len(tick_labels) + 1))
    ax.set_xticklabels(tick_labels, rotation=0)
    ax.set_title(title_text, fontsize=11)
    ax.set_xlabel("IR")
    ax.set_ylabel("Feature set entropy (nats)")
    ax.grid(True, axis='y', alpha=0.3)

# ---------- Plot grid ----------
def plot_violin_entropy_grid(df: pd.DataFrame, dataset: str, k_max: int,
                             unified_y: bool = False, save_to_file: bool = False):
    ds_df = df[df['dataset'] == dataset]
    dataset_display = DISPLAY_NAMES.get(dataset, dataset)

    if ds_df.empty:
        print(f"[WARN] No data for dataset: {dataset_display}")
        return

    models = sorted(ds_df['method'].dropna().unique().tolist())
    if not models:
        print("[WARN] No models found for the selected dataset.")
        return

    # precompute curves for all models + global y-limits
    model_to_curves = {}
    global_y_max = 0.0
    all_present_irs = set()

    for m in models:
        curves = compute_entropy_by_IR_over_k(df, dataset, m, k_max)
        model_to_curves[m] = curves
        if curves.empty:
            continue
        all_present_irs.update(curves["IR"].unique().tolist())
        ymax = float(np.nanmax(curves['entropy'].to_numpy()))
        global_y_max = max(global_y_max, ymax)

    n_models = len(models)
    fig_height = max(3.8 * n_models, 4.0)
    fig, axes = plt.subplots(nrows=n_models, ncols=1, figsize=(10, fig_height), sharey=unified_y)
    if n_models == 1:
        axes = [axes]

    fig.suptitle(f"{dataset_display} — Feature set entropy across IRs (violin, all models)",
                 y=0.995, fontsize=13)

    # unified y-axis range (entropy >= 0)
    if unified_y:
        # prosta górna granica: zaokrągl do 1 miejsca po przecinku i dodaj niewielki margines
        y_hi = float(np.ceil(max(global_y_max, 0.01) * 10.0) / 10.0) + 1e-6
        for ax in axes:
            ax.set_ylim(0.0, y_hi)

    for ax, m in zip(axes, models):
        curves = model_to_curves[m]
        if curves is None or curves.empty:
            ax.text(0.5, 0.5, "No data", ha='center', va='center')
            ax.set_title(f"Model: {m}")
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            continue

        data_by_ir, tick_labels, body_colors, present_irs = _collect_violin_data_from_entropy(curves)
        if not data_by_ir:
            ax.text(0.5, 0.5, "No IR data", ha='center', va='center')
            ax.set_title(f"Model: {m}")
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            continue

        _draw_single_violin(ax, data_by_ir, tick_labels, body_colors, title_text=f"Model: {m}")

        if not unified_y:
            # lokalne granice osi Y (>=0) z niewielką poduszką
            y_min = 0.0
            y_max = float(np.nanmax(np.concatenate(data_by_ir)))
            pad = max(1e-3, (y_max - y_min) * 0.1)
            ax.set_ylim(y_min, y_max + pad)

    # Global legend of IR colors at the bottom
    legend_handles = [
        Patch(facecolor=IR_COLORS.get(ir, 'gray'), edgecolor='black',
              label=IR_LABELS.get(ir, str(ir)))
        for ir in IR_ORDER if ir in all_present_irs
    ]
    if legend_handles:
        fig.legend(handles=legend_handles,
                   loc='lower center',
                   ncol=min(len(legend_handles), 6),
                   bbox_to_anchor=(0.5, -0.02),
                   frameon=False)

    fig.tight_layout(rect=[0, 0.04, 1, 0.97])
    plt.show()

    # Save
    if save_to_file:
        fn = f"{dataset_display}__ALL_MODELS__Entropy_Violin_GRID1x{n_models}.{FIG_FMT}"
        out_path = os.path.join(OUT_DIR_VIOLIN_ENT, fn)
        fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)
        print(f"[INFO] Saved: {out_path}")

# ---------- Widgets ----------
w_dsVE = Dropdown(
    options=[(DISPLAY_NAMES.get(ds, ds), ds) for ds in dataset_identifiers],
    value=dataset_identifiers[0] if dataset_identifiers else None,
    description="Dataset:",
    layout=Layout(width="260px")
)

w_kmaxVE = IntSlider(
    value=5, min=1, max=10, step=1,
    description="Top-N (max K):",
    readout=True, readout_format='d',
    continuous_update=False,
    layout=Layout(width="350px")
)

w_unifiedYVE = Checkbox(value=False, description="Unified Y axis")
w_saveVE     = Checkbox(value=False, description="Save to file")
outVE = Output()

w_kmaxVE.style = {"description_width": "initial"}

def _recompute_violin_entropy(dataset, k_max, unified_y, save_to_file):
    outVE.clear_output(wait=True)
    with outVE:
        plot_violin_entropy_grid(long_df, dataset, k_max, unified_y=unified_y, save_to_file=save_to_file)

controlsVE = {
    'dataset': w_dsVE,
    'k_max': w_kmaxVE,
    'unified_y': w_unifiedYVE,
    'save_to_file': w_saveVE
}

row1ve = HBox([w_dsVE], layout=Layout(align_items="center", gap="16px"))
row2ve = HBox([w_kmaxVE, w_unifiedYVE, w_saveVE], layout=Layout(align_items="center", gap="16px"))
ioVE = interactive_output(_recompute_violin_entropy, controlsVE)

display(VBox([row1ve, row2ve, outVE], layout=Layout(gap="8px")))

# Initial draw
if dataset_identifiers:
    _recompute_violin_entropy(w_dsVE.value, w_kmaxVE.value, w_unifiedYVE.value, w_saveVE.value)
else:
    print("[WARN] No available datasets in 'dataset_identifiers'.")


# Average Jaccard Similarity across IRs

In [10]:
# ============================================
# Average Jaccard Similarity of Top-N feature sets across subsets for each IR
# - full, self-contained Jupyter cell
# - shows ALL models simultaneously in a 1×N grid (one column, N rows)
# - independent toggles: "Unified Y axis" and "Unified X axis"
# - optional saving to a single combined figure
# - uses the same globals (IR_ORDER/LABELS/COLORS/DISPLAY_NAMES/FIG_FMT/FIG_DPI/base_path)
#   and the same dataset selector style as the base cell
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, Layout, interactive_output
from IPython.display import display
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.spatial.distance import pdist, squareform

# ---------- Required input ----------
try:
    assert isinstance(long_df, pd.DataFrame)
except Exception as _e:
    raise RuntimeError("This cell requires 'long_df' in memory (pd.DataFrame). Define it earlier.") from _e

# ---------- IR order/labels/colors fallbacks ----------
if 'IR_ORDER' not in globals():
    IR_ORDER = sorted(long_df['IR'].dropna().unique().tolist(), key=lambda x: (str(type(x)), x))
if 'IR_LABELS' not in globals():
    IR_LABELS = {ir: str(ir) for ir in IR_ORDER}
if 'IR_COLORS' not in globals():
    cmap = plt.get_cmap('tab20')
    IR_COLORS = {ir: cmap(i % 20) for i, ir in enumerate(IR_ORDER)}

# ---------- Display name map / figure params / base path fallbacks ----------
if 'DISPLAY_NAMES' not in globals():
    DISPLAY_NAMES = {}
if 'FIG_FMT' not in globals():
    FIG_FMT = 'png'
if 'FIG_DPI' not in globals():
    FIG_DPI = 160
if 'base_path' not in globals():
    base_path = '.'

# ---------- Dataset identifiers for the selector ----------
if 'dataset_identifiers' not in globals():
    dataset_identifiers = sorted(long_df['dataset'].dropna().unique().tolist())

# ---------- Output directory ----------
OUT_DIR_JACCARD_GRID = os.path.join(base_path, "figs_TOPsets_avgJaccard_byIR_GRID1xN")
os.makedirs(OUT_DIR_JACCARD_GRID, exist_ok=True)

# ---------- Utilities ----------
def _clamp(n: int, lo: int = 1, hi: int = 10) -> int:
    try:
        n = int(n)
    except Exception:
        n = lo
    return int(max(lo, min(hi, n)))

# ---------- Core: build ranked feature lists per (method × subset × IR) ----------
def compute_ranked_features_per_bin(df: pd.DataFrame, dataset: str, model: str) -> pd.DataFrame:
    """
    For a selected dataset and model, compute the full ranking of features (by mean abs shapMean)
    within each (subset, IR). Returns DataFrame:
        [IR, IR_label, subset_filled, ranked_features (tuple of features in desc. importance)]
    """
    g = df[(df['dataset'] == dataset) & (df['method'] == model)].copy()
    if g.empty:
        return pd.DataFrame(columns=["IR", "IR_label", "subset_filled", "ranked_features"])

    g['subset_filled'] = g['subset'].fillna(-1)

    recs = []
    for (subset_val, ir), sub in g.groupby(['subset_filled', 'IR'], dropna=False):
        imp = (sub.assign(a=sub['shapMean'].abs())
                 .groupby('feature', as_index=False)['a'].mean()
                 .sort_values('a', ascending=False))
        ranked = tuple(imp['feature'].tolist())
        if len(ranked) == 0:
            continue
        recs.append({
            "IR": ir,
            "IR_label": IR_LABELS.get(ir, str(ir)),
            "subset_filled": subset_val,
            "ranked_features": ranked
        })

    out = pd.DataFrame(recs)
    if out.empty:
        return out

    out["IR_order_idx"] = out["IR"].apply(lambda r: IR_ORDER.index(r) if r in IR_ORDER else 999)
    return out.sort_values(["IR_order_idx", "subset_filled"]).drop(columns="IR_order_idx")

def average_jaccard_for_k(feature_orders: list[tuple], k: int) -> float:
    """
    Given a list of full rankings (tuples) for multiple subsets, compute the
    average pairwise Jaccard similarity of their Top-k sets.
    """
    if not feature_orders:
        return np.nan
    if len(feature_orders) == 1:
        return 1.0  # single subset ⇒ similarity = 1

    # Build Top-k sets
    sets_k = [set(r[:k]) for r in feature_orders]

    # MultiLabelBinarizer to consistent space
    mlb = MultiLabelBinarizer()
    bin_mat = mlb.fit_transform(sets_k)

    # Pairwise Jaccard distance → similarity
    # If any set is empty (k==0), this would be degenerate; k is clamped to >=1.
    dist = pdist(bin_mat, metric='jaccard')  # distances in [0,1]
    if dist.size == 0:
        return 1.0
    sim = 1.0 - dist  # convert to similarities
    return float(sim.mean())

def compute_avg_jaccard_curves(df: pd.DataFrame, dataset: str, model: str, k_max: int) -> pd.DataFrame:
    """
    For given dataset and model, returns tidy DF with columns:
        [IR, IR_label, k, avg_jaccard]
    where 'k' runs 1..k_max.
    """
    k_max = _clamp(k_max, lo=1, hi=10)

    ranks_df = compute_ranked_features_per_bin(df, dataset, model)
    if ranks_df.empty:
        return pd.DataFrame(columns=["IR", "IR_label", "k", "avg_jaccard"])

    out = []
    for ir, sub in ranks_df.groupby("IR"):
        orders = sub["ranked_features"].tolist()
        for k in range(1, k_max + 1):
            aj = average_jaccard_for_k(orders, k)
            out.append({
                "IR": ir,
                "IR_label": IR_LABELS.get(ir, str(ir)),
                "k": k,
                "avg_jaccard": aj
            })
    res = pd.DataFrame(out)
    res["IR_order_idx"] = res["IR"].apply(lambda r: IR_ORDER.index(r) if r in IR_ORDER else 999)
    return res.sort_values(["IR_order_idx", "k"]).drop(columns="IR_order_idx")

# ---------- Plotting (1×N grid, one subplot per model) ----------
def plot_avg_jaccard_lines_grid(df: pd.DataFrame, dataset: str, k_max: int,
                                unified_y: bool = False, unified_x: bool = False,
                                save_to_file: bool = False):
    """
    Builds a 1×N grid (N rows, 1 column) where each subplot shows curves (IR lines)
    of Average Jaccard Similarity vs Top-N for a single model.
    """
    ds_df = df[df['dataset'] == dataset]
    if ds_df.empty:
        ds_name = DISPLAY_NAMES.get(dataset, dataset)
        print(f"[WARN] No data for dataset: {ds_name}")
        return

    models = sorted(ds_df['method'].dropna().unique().tolist())
    if not models:
        print("[WARN] No models found for the selected dataset.")
        return

    # Precompute curves and global limits if needed
    model_to_curves = {}
    global_y_min, global_y_max = 1.0, 0.0
    global_x_max = 0
    for m in models:
        curves = compute_avg_jaccard_curves(df, dataset, m, k_max)
        model_to_curves[m] = curves
        if not curves.empty:
            ymax = float(np.nanmax(curves['avg_jaccard'].to_numpy()))
            ymin = float(np.nanmin(curves['avg_jaccard'].to_numpy()))
            global_y_max = max(global_y_max, ymax)
            global_y_min = min(global_y_min, ymin)
            global_x_max = max(global_x_max, int(curves['k'].max()))

    # Prepare figure
    n_models = len(models)
    fig_height = max(4.0 * n_models, 4.0)
    fig, axes = plt.subplots(
        nrows=n_models, ncols=1,
        figsize=(10, fig_height),
        sharex=unified_x, sharey=unified_y
    )
    if n_models == 1:
        axes = [axes]

    dataset_display = DISPLAY_NAMES.get(dataset, dataset)
    fig.suptitle(
        f"{dataset_display} — Average Jaccard Similarity of Top-N sets (all models)",
        y=0.995, fontsize=12
    )

    # Unified ticks/limits
    if unified_x:
        if global_x_max <= 0:
            global_x_max = _clamp(k_max, lo=1, hi=10)
        x_ticks = np.arange(1, global_x_max + 1, 1)

    if unified_y:
        # Keep it within [0,1] but adapt to the actual range
        y_lo = max(0.0, np.floor(global_y_min * 10) / 10.0)
        y_hi = min(1.0, np.ceil(global_y_max * 10) / 10.0)
        if y_lo == y_hi:
            y_lo = max(0.0, y_lo - 0.1)
            y_hi = min(1.0, y_hi + 0.1)
        y_ticks = np.linspace(y_lo, y_hi, num=6)

    # Draw per-model subplot
    for ax, m in zip(axes, models):
        curves = model_to_curves[m]
        if curves.empty:
            ax.text(0.5, 0.5, "No data", ha='center', va='center')
            ax.set_title(f"Model: {m}")
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            continue

        present_irs = [ir for ir in IR_ORDER if ir in curves["IR"].unique()] or curves["IR"].unique().tolist()
        for ir in present_irs:
            sub = curves[curves["IR"] == ir]
            if sub.empty:
                continue
            xs = sub["k"].to_numpy()
            ys = sub["avg_jaccard"].to_numpy(dtype=float)
            label = IR_LABELS.get(ir, str(ir))
            color = IR_COLORS.get(ir, None)
            ax.plot(xs, ys, marker='o', linestyle='-', label=label, color=color)

        ax.set_title(f"Model: {m}")
        ax.set_xlabel("Top-N Features (k)")
        ax.set_ylabel("Average Jaccard Similarity")

        if unified_x:
            ax.set_xticks(x_ticks)
            ax.set_xlim(left=1, right=max(1, global_x_max))
        else:
            unique_xs = sorted(curves["k"].unique().tolist())
            ax.set_xticks(unique_xs)
            if unique_xs:
                ax.set_xlim(left=min(unique_xs), right=max(unique_xs))

        if unified_y:
            ax.set_yticks(y_ticks)
            ax.set_ylim(bottom=y_ticks[0], top=y_ticks[-1])
        else:
            y_min = float(np.nanmin(curves["avg_jaccard"].to_numpy()))
            y_max = float(np.nanmax(curves["avg_jaccard"].to_numpy()))
            if not np.isfinite(y_min) or not np.isfinite(y_max):
                y_min, y_max = 0.0, 1.0
            pad = max(0.02, (y_max - y_min) * 0.1)
            ax.set_ylim(bottom=max(0.0, y_min - pad), top=min(1.0, y_max + pad))

        ax.legend(title="Imbalance Ratios", loc="lower right")
        ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

    # Optional save-to-file (rebuild clean figure for export)
    if save_to_file:
        fn = f"{dataset_display}__ALL_MODELS__AvgJaccard_GRID1x{n_models}" \
             f"{'_unifiedY' if unified_y else ''}{'_unifiedX' if unified_x else ''}. {FIG_FMT}".replace('..','.')
        out_path = os.path.join(OUT_DIR_JACCARD_GRID, fn)

        fig, axes = plt.subplots(
            nrows=n_models, ncols=1,
            figsize=(10, fig_height),
            sharex=unified_x, sharey=unified_y
        )
        if n_models == 1:
            axes = [axes]
        fig.suptitle(
            f"{dataset_display} — Average Jaccard Similarity of Top-N sets (all models)",
            y=0.995, fontsize=12
        )

        if unified_x:
            if global_x_max <= 0:
                global_x_max = _clamp(k_max, lo=1, hi=10)
            x_ticks = np.arange(1, global_x_max + 1, 1)
        if unified_y:
            y_lo = max(0.0, np.floor(global_y_min * 10) / 10.0)
            y_hi = min(1.0, np.ceil(global_y_max * 10) / 10.0)
            if y_lo == y_hi:
                y_lo = max(0.0, y_lo - 0.1)
                y_hi = min(1.0, y_hi + 0.1)
            y_ticks = np.linspace(y_lo, y_hi, num=6)

        for ax, m in zip(axes, models):
            curves = model_to_curves[m]
            if curves.empty:
                ax.text(0.5, 0.5, "No data", ha='center', va='center')
                ax.set_title(f"Model: {m}")
                ax.grid(axis='y', linestyle='--', alpha=0.7)
                continue

            present_irs = [ir for ir in IR_ORDER if ir in curves["IR"].unique()] or curves["IR"].unique().tolist()
            for ir in present_irs:
                sub = curves[curves["IR"] == ir]
                xs = sub["k"].to_numpy()
                ys = sub["avg_jaccard"].to_numpy(dtype=float)
                label = IR_LABELS.get(ir, str(ir))
                color = IR_COLORS.get(ir, None)
                ax.plot(xs, ys, marker='o', linestyle='-', label=label, color=color)

            ax.set_title(f"Model: {m}")
            ax.set_xlabel("Top-N Features (k)")
            ax.set_ylabel("Average Jaccard Similarity")

            if unified_x:
                ax.set_xticks(x_ticks)
                ax.set_xlim(left=1, right=max(1, global_x_max))
            else:
                unique_xs = sorted(curves["k"].unique().tolist())
                ax.set_xticks(unique_xs)
                if unique_xs:
                    ax.set_xlim(left=min(unique_xs), right=max(unique_xs))

            if unified_y:
                ax.set_yticks(y_ticks)
                ax.set_ylim(bottom=y_ticks[0], top=y_ticks[-1])
            else:
                y_min = float(np.nanmin(curves["avg_jaccard"].to_numpy()))
                y_max = float(np.nanmax(curves["avg_jaccard"].to_numpy()))
                if not np.isfinite(y_min) or not np.isfinite(y_max):
                    y_min, y_max = 0.0, 1.0
                pad = max(0.02, (y_max - y_min) * 0.1)
                ax.set_ylim(bottom=max(0.0, y_min - pad), top=min(1.0, y_max + pad))

            ax.legend(title="Imbalance Ratios", loc="lower right")
            ax.grid(axis='y', linestyle='--', alpha=0.7)

        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.close()
        print(f"[INFO] Saved: {out_path}")

# ---------- Widgets (dataset + controls; no model selector because we show all) ----------
w_dsJ = Dropdown(
    options=[(DISPLAY_NAMES.get(ds, ds), ds) for ds in dataset_identifiers],
    value=dataset_identifiers[0] if dataset_identifiers else None,
    description="Dataset:",
    layout=Layout(width="260px")
)

w_kmaxJ = IntSlider(
    value=5, min=1, max=10, step=1,
    description="Top-N (max):",
    readout=True, readout_format='d',
    continuous_update=False,
    layout=Layout(width="350px")
)

w_unifiedYJ = Checkbox(value=False, description="Unified Y axis")
w_unifiedXJ = Checkbox(value=False, description="Unified X axis")
w_saveJ     = Checkbox(value=False, description="Save to file")
outJ = Output()

# A bit nicer label alignment
w_kmaxJ.style = {"description_width": "initial"}

def _recompute_jaccard_grid(dataset, k_max, unified_y, unified_x, save_to_file):
    outJ.clear_output(wait=True)
    with outJ:
        plot_avg_jaccard_lines_grid(
            long_df, dataset, k_max,
            unified_y=unified_y, unified_x=unified_x,
            save_to_file=save_to_file
        )

controlsJ = {
    'dataset': w_dsJ,
    'k_max': w_kmaxJ,
    'unified_y': w_unifiedYJ,
    'unified_x': w_unifiedXJ,
    'save_to_file': w_saveJ
}

row1j = HBox([w_dsJ], layout=Layout(align_items="center", gap="16px"))
row2j = HBox([w_kmaxJ, w_unifiedYJ, w_unifiedXJ, w_saveJ], layout=Layout(align_items="center", gap="16px"))
ioJ = interactive_output(_recompute_jaccard_grid, controlsJ)

display(VBox([row1j, row2j, outJ], layout=Layout(gap="8px")))

# Initial draw
if dataset_identifiers:
    _recompute_jaccard_grid(w_dsJ.value, w_kmaxJ.value, w_unifiedYJ.value, w_unifiedXJ.value, w_saveJ.value)
else:
    print("[WARN] No available datasets in 'dataset_identifiers'.")


In [12]:
# ============================================
# Violin plots: Average Jaccard Similarity across IRs (all models; 1×N vertical)
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, Checkbox, HBox, VBox, Output, Layout, interactive_output
from IPython.display import display
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.spatial.distance import pdist
from matplotlib.patches import Patch

# ---------- Required input ----------
try:
    assert isinstance(long_df, pd.DataFrame)
except Exception as _e:
    raise RuntimeError("This cell requires 'long_df' in memory (pd.DataFrame).") from _e

_required_cols = {'dataset','method','subset','IR','feature','shapMean'}
missing_cols = _required_cols - set(long_df.columns)
if missing_cols:
    raise RuntimeError(f"'long_df' is missing required columns: {sorted(missing_cols)}")

# ---------- Fallbacks ----------
if 'IR_ORDER' not in globals():
    IR_ORDER = sorted(long_df['IR'].dropna().unique().tolist())
if 'IR_LABELS' not in globals():
    IR_LABELS = {ir: str(ir) for ir in IR_ORDER}
if 'IR_COLORS' not in globals():
    cmap = plt.get_cmap('tab20')
    IR_COLORS = {ir: cmap(i % 20) for i, ir in enumerate(IR_ORDER)}
if 'DISPLAY_NAMES' not in globals():
    DISPLAY_NAMES = {}
if 'FIG_FMT' not in globals():
    FIG_FMT = 'png'
if 'FIG_DPI' not in globals():
    FIG_DPI = 160
if 'base_path' not in globals():
    base_path = '.'

if 'dataset_identifiers' not in globals():
    dataset_identifiers = sorted(long_df['dataset'].dropna().unique().tolist())

OUT_DIR_VIOLIN_JACCARD = os.path.join(base_path, "figs_AvgJaccard_violin_byIR_GRID1xN")
os.makedirs(OUT_DIR_VIOLIN_JACCARD, exist_ok=True)

# ---------- Helpers ----------
def _clamp(n, lo=1, hi=10):
    try:
        n = int(n)
    except Exception:
        n = lo
    return int(max(lo, min(hi, n)))

def compute_ranked_features_per_bin(df, dataset, model):
    g = df[(df['dataset'] == dataset) & (df['method'] == model)].copy()
    if g.empty:
        return pd.DataFrame(columns=["IR","IR_label","subset_filled","ranked_features"])
    g['subset_filled'] = g['subset'].fillna(-1)

    recs = []
    for (subset_val, ir), sub in g.groupby(['subset_filled','IR'], dropna=False):
        imp = (sub.assign(a=sub['shapMean'].abs())
                 .groupby('feature', as_index=False)['a'].mean()
                 .sort_values('a', ascending=False))
        ranked = tuple(imp['feature'].tolist())
        if ranked:
            recs.append({
                "IR": ir,
                "IR_label": IR_LABELS.get(ir, str(ir)),
                "subset_filled": subset_val,
                "ranked_features": ranked
            })
    return pd.DataFrame(recs)

def average_jaccard_for_k(feature_orders, k):
    if not feature_orders:
        return np.nan
    if len(feature_orders) == 1:
        return 1.0
    sets_k = [set(r[:k]) for r in feature_orders]
    mlb = MultiLabelBinarizer()
    bin_mat = mlb.fit_transform(sets_k)
    dist = pdist(bin_mat, metric='jaccard')
    if dist.size == 0:
        return 1.0
    return float((1.0 - dist).mean())

def compute_avg_jaccard_by_IR_over_k(df, dataset, model, k_max):
    k_max = _clamp(k_max)
    ranks = compute_ranked_features_per_bin(df, dataset, model)
    if ranks.empty:
        return pd.DataFrame(columns=["IR","IR_label","k","avg_jaccard"])
    recs = []
    for ir, sub in ranks.groupby("IR"):
        orders = sub["ranked_features"].tolist()
        for k in range(1, k_max+1):
            recs.append({
                "IR": ir,
                "IR_label": IR_LABELS.get(ir, str(ir)),
                "k": k,
                "avg_jaccard": average_jaccard_for_k(orders, k)
            })
    return pd.DataFrame(recs)

def _collect_violin_data_from_curves(curves_df):
    if curves_df is None or curves_df.empty:
        return [], [], [], []
    present_irs = [ir for ir in IR_ORDER if ir in curves_df["IR"].unique()]
    if not present_irs:
        present_irs = curves_df["IR"].unique().tolist()
    data_by_ir, tick_labels, body_colors = [], [], []
    for ir in present_irs:
        sub = curves_df[curves_df["IR"] == ir].sort_values("k")
        ys = sub["avg_jaccard"].to_numpy(dtype=float)
        if ys.size == 0:
            continue
        data_by_ir.append(ys)
        tick_labels.append(IR_LABELS.get(ir, str(ir)))
        body_colors.append(IR_COLORS.get(ir, None))
    return data_by_ir, tick_labels, body_colors, present_irs

def _draw_single_violin(ax, data_by_ir, tick_labels, body_colors, title_text):
    parts = ax.violinplot(
        dataset=data_by_ir,
        showmeans=True,
        showmedians=True,
        showextrema=True
    )
    for i, b in enumerate(parts['bodies']):
        col = body_colors[i]
        if col:
            b.set_facecolor(col)
        b.set_alpha(0.7)
        b.set_edgecolor('black')
        b.set_linewidth(0.8)
    for k in ('cmeans', 'cmedians', 'cbars', 'cmins', 'cmaxes'):
        if k in parts:
            parts[k].set_linewidth(1.2)
            parts[k].set_color('black')
    ax.set_xticks(np.arange(1, len(tick_labels)+1))
    ax.set_xticklabels(tick_labels)
    ax.set_title(title_text)
    ax.set_xlabel("IR")
    ax.set_ylabel("Average Jaccard Similarity")
    ax.grid(axis='y', alpha=0.3)

def plot_violin_jaccard_grid(df, dataset, k_max, unified_y=False, save_to_file=False):
    ds_df = df[df['dataset'] == dataset]
    dataset_display = DISPLAY_NAMES.get(dataset, dataset)
    if ds_df.empty:
        print(f"[WARN] No data for dataset {dataset_display}")
        return

    models = sorted(ds_df['method'].dropna().unique().tolist())
    if not models:
        print("[WARN] No models found for dataset.")
        return

    model_to_curves = {}
    all_present_irs = set()
    for m in models:
        curves = compute_avg_jaccard_by_IR_over_k(df, dataset, m, k_max)
        model_to_curves[m] = curves
        if not curves.empty:
            all_present_irs.update(curves["IR"].unique().tolist())

    fig_height = max(3.8 * len(models), 4.0)
    fig, axes = plt.subplots(nrows=len(models), ncols=1, figsize=(10, fig_height), sharey=unified_y)
    if len(models) == 1:
        axes = [axes]

    fig.suptitle(f"{dataset_display} — Average Jaccard Similarity (violin, all models)", fontsize=13)

    for ax, m in zip(axes, models):
        curves = model_to_curves[m]
        if curves is None or curves.empty:
            ax.text(0.5, 0.5, "No data", ha='center', va='center')
            ax.set_title(f"Model: {m}")
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            continue
        data_by_ir, tick_labels, body_colors, present_irs = _collect_violin_data_from_curves(curves)
        if not data_by_ir:
            ax.text(0.5, 0.5, "No IR data", ha='center', va='center')
            ax.set_title(f"Model: {m}")
            continue
        _draw_single_violin(ax, data_by_ir, tick_labels, body_colors, f"Model: {m}")
        if unified_y:
            ax.set_ylim(0, 1)

    # FIXED HERE: only two closing parentheses
    legend_handles = [
        Patch(facecolor=IR_COLORS.get(ir, 'gray'),
              edgecolor='black',
              label=IR_LABELS.get(ir, str(ir)))
        for ir in IR_ORDER if ir in all_present_irs
    ]

    if legend_handles:
        fig.legend(handles=legend_handles, loc='lower center', ncol=min(len(legend_handles), 6),
                   bbox_to_anchor=(0.5, -0.02), frameon=False)

    fig.tight_layout(rect=[0, 0.04, 1, 0.97])
    plt.show()

    if save_to_file:
        out_path = os.path.join(OUT_DIR_VIOLIN_JACCARD,
                                f"{dataset_display}__AvgJaccard_Violin_ALL_MODELS.{FIG_FMT}")
        fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
        plt.close(fig)
        print(f"[INFO] Saved: {out_path}")

# ---------- Widgets ----------
w_dsVJ = Dropdown(
    options=[(DISPLAY_NAMES.get(ds, ds), ds) for ds in dataset_identifiers],
    value=dataset_identifiers[0] if dataset_identifiers else None,
    description="Dataset:",
    layout=Layout(width="260px")
)

w_kmaxVJ = IntSlider(value=5, min=1, max=10, step=1,
                     description="Top-N (max):", layout=Layout(width="300px"))
w_unifiedYVJ = Checkbox(value=True, description="Unified Y axis (0–1)")
w_saveVJ = Checkbox(value=False, description="Save to file")
outVJ = Output()

def _update_violin(dataset, k_max, unified_y, save_to_file):
    outVJ.clear_output(wait=True)
    with outVJ:
        plot_violin_jaccard_grid(long_df, dataset, k_max, unified_y, save_to_file)

controls = {'dataset': w_dsVJ, 'k_max': w_kmaxVJ, 'unified_y': w_unifiedYVJ, 'save_to_file': w_saveVJ}
ui = VBox([
    HBox([w_dsVJ]),
    HBox([w_kmaxVJ, w_unifiedYVJ, w_saveVJ]),
    outVJ
])
display(ui)
interactive_output(_update_violin, controls)

# Initial draw
if dataset_identifiers:
    _update_violin(w_dsVJ.value, w_kmaxVJ.value, w_unifiedYVJ.value, w_saveVJ.value)
else:
    print("[WARN] No available datasets in 'dataset_identifiers'.")
